# Evaluation of precomputed full-time-series archives

This notebook reads existing reference, standard M3C2, and TAM3C2 archives and evaluates their stored 4D-OBC results. It does not rerun distance estimation or object extraction.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
import py4dgeo

## 1. Configuration

In [ ]:
# Folder created by the scenario-specific producer notebook.
# It may be an absolute path or a folder relative to this notebook.
archive_folder = os.path.join(
    os.getcwd(),
    "simulation_test2_als_downsampled1",
)

# Each specification may be an exact filename, an absolute ZIP path, or a glob
# pattern. A glob pattern must resolve to exactly one archive.
# Normal and spatial-gap producer notebooks normally use these two patterns.
tam3c2_archive_spec = "*_full_timeseries_best_scale_idx*_weighted_tam3c2.zip"
m3c2_archive_spec = "*_full_timeseries_best_scale_idx*_standard_m3c2.zip"

# For the temporal-gap notebook that reconstructs withheld timestamps, use:
# tam3c2_archive_spec = "*_full_timeline_reconstructed_best_scale_idx*_weighted_tam3c2.zip"
# m3c2_archive_spec = "*_observed_epochs_only_best_scale_idx*_standard_m3c2.zip"

# The mesh-derived reference archive is independent of the selected scenario.
reference_archive_path = os.path.join(
    os.getcwd(),
    "simulation2_reference.zip",
)

# Object-level matching threshold used by the evaluation below.
match_iou_threshold = 0.80


## 2. Load precomputed analysis archives


In [ ]:
def resolve_archive(folder, archive_spec, label):
    folder = os.path.abspath(folder)
    if not os.path.isdir(folder):
        raise FileNotFoundError(f"Archive folder not found: {folder}")

    candidate = (
        archive_spec
        if os.path.isabs(archive_spec)
        else os.path.join(folder, archive_spec)
    )

    if glob.has_magic(candidate):
        matches = sorted(glob.glob(candidate))

        # If the specified folder is a parent directory, also search below it.
        if not matches and not os.path.isabs(archive_spec):
            recursive_pattern = os.path.join(folder, "**", archive_spec)
            matches = sorted(glob.glob(recursive_pattern, recursive=True))

        matches = [match for match in matches if os.path.isfile(match)]
        if len(matches) != 1:
            available_zip_files = sorted(
                glob.glob(os.path.join(folder, "**", "*.zip"), recursive=True)
            )
            available_text = "\n".join(
                f"  - {zip_path}" for zip_path in available_zip_files
            ) or "  (no ZIP files found)"
            raise ValueError(
                f"{label} archive pattern must match exactly one file; "
                f"found {len(matches)} matches for {archive_spec!r} under {folder!r}.\n"
                f"Available ZIP files:\n{available_text}"
            )
        candidate = matches[0]

    if not os.path.isfile(candidate):
        available_zip_files = sorted(
            glob.glob(os.path.join(folder, "**", "*.zip"), recursive=True)
        )
        available_text = "\n".join(
            f"  - {zip_path}" for zip_path in available_zip_files
        ) or "  (no ZIP files found)"
        raise FileNotFoundError(
            f"{label} archive not found: {candidate}\n"
            f"Available ZIP files under {folder}:\n{available_text}"
        )
    return os.path.abspath(candidate)

gt_path = os.path.abspath(reference_archive_path)
if not os.path.isfile(gt_path):
    raise FileNotFoundError(f"Reference archive not found: {gt_path}")

tam3c2_path = resolve_archive(
    archive_folder,
    tam3c2_archive_spec,
    "TAM3C2",
)
m3c2_path = resolve_archive(
    archive_folder,
    m3c2_archive_spec,
    "M3C2",
)

ref_analysis = py4dgeo.SpatiotemporalAnalysis(gt_path, force=False)
analysis = py4dgeo.SpatiotemporalAnalysis(tam3c2_path, force=False)
m3c2_analysis = py4dgeo.SpatiotemporalAnalysis(m3c2_path, force=False)

analyses = {
    "GT": ref_analysis,
    "M3C2": m3c2_analysis,
    "TAM3C2": analysis,
}

for name, st_analysis in analyses.items():
    if st_analysis.objects is None:
        raise RuntimeError(
            f"{name} archive contains no stored 4D-OBC objects: "
            f"{st_analysis.filename}. Run object extraction in the producer notebook first."
        )
    print(
        f"{name}: {len(st_analysis.objects)} objects, "
        f"{st_analysis.corepoints.cloud.shape[0]:,} corepoints, "
        f"distance shape={st_analysis.distances.shape}, "
        f"path={st_analysis.filename}"
    )


## 9. Compare GT, M3C2, and TAM3C2 4D-OBCs

The comparison treats every 4D-OBC as a Cartesian product of its core-point set and its continuous timestamp interval. Object matching is one-to-one and maximizes total spatiotemporal IoU before applying the match threshold.

In [ ]:
# The archives were loaded in Section 2. Keep the shared dictionary interface
# used by the evaluation code below.
analyses = {
    'GT': ref_analysis,
    'M3C2': m3c2_analysis,
    'TAM3C2': analysis,
}

for name, st_analysis in analyses.items():
    objects_for_method = st_analysis.objects
    if objects_for_method is None:
        raise RuntimeError(
            f"{name} analysis has no stored 4D-OBC objects. "
            "Run RegionGrowingAlgorithm in the producer notebook first."
        )
    print(
        f"{name}: {len(objects_for_method)} objects, "
        f"{st_analysis.corepoints.cloud.shape[0]:,} corepoints, "
        f"distance shape={st_analysis.distances.shape}, "
        f"path={st_analysis.filename}"
    )


In [ ]:
from scipy.optimize import linear_sum_assignment

try:
    import pandas as pd
except ImportError:
    pd = None


_IOU_TOL = 1e-12


def _timestamps_for_analysis(st_analysis):
    timedeltas = list(st_analysis.timedeltas)
    if len(timedeltas) != st_analysis.distances.shape[1]:
        raise ValueError(
            f"{st_analysis.filename}: {len(timedeltas)} timestamps for "
            f"{st_analysis.distances.shape[1]} distance epochs"
        )

    reference_time = st_analysis.reference_epoch.timestamp
    timestamps = np.array([reference_time + td for td in timedeltas], dtype=object)
    if len(timestamps) == 0:
        raise ValueError(f"{st_analysis.filename}: no acquisition timestamps available")

    time_days = np.array(
        [(ts - timestamps[0]).total_seconds() / 86400.0 for ts in timestamps],
        dtype=float,
    )
    return timestamps, time_days


def _validate_common_corepoints(analyses):
    base_name = 'GT'
    base_corepoints = np.asarray(analyses[base_name].corepoints.cloud)
    for name, st_analysis in analyses.items():
        corepoints_for_method = np.asarray(st_analysis.corepoints.cloud)
        if corepoints_for_method.shape != base_corepoints.shape or not np.allclose(
            corepoints_for_method,
            base_corepoints,
        ):
            raise ValueError(
                f"{name} corepoints differ from {base_name}; this evaluation assumes "
                "identical core-point arrays and indices."
            )
    print(
        f"All analyses use the same {base_corepoints.shape[0]:,} corepoints; "
        "nearest-neighbor corepoint mapping is bypassed."
    )


def _object_properties(obj, time_days):
    corepoints_for_object = set(int(idx) for idx in np.asarray(obj.indices, dtype=int))
    start_epoch = int(obj.start_epoch)
    end_epoch = int(obj.end_epoch)

    if start_epoch > end_epoch:
        raise ValueError(f"Invalid object interval: {start_epoch} > {end_epoch}")
    if start_epoch < 0 or end_epoch >= len(time_days):
        raise IndexError(
            f"Object epoch interval [{start_epoch}, {end_epoch}] outside "
            f"timestamp array of length {len(time_days)}"
        )
    if not corepoints_for_object:
        raise ValueError("Object has empty corepoint support")

    start_time = float(time_days[start_epoch])
    end_time = float(time_days[end_epoch])
    duration = end_time - start_time
    if duration < 0:
        raise ValueError(f"Object has negative duration: {duration}")

    return {
        'corepoints': corepoints_for_object,
        'start_epoch': start_epoch,
        'end_epoch': end_epoch,
        'start_time': start_time,
        'end_time': end_time,
        'duration': duration,
        'n_corepoints': len(corepoints_for_object),
    }


def _pairwise_object_metrics(det_obj, ref_obj, det_time_days, ref_time_days):
    det = _object_properties(det_obj, det_time_days)
    ref = _object_properties(ref_obj, ref_time_days)

    shared_corepoint_count = len(det['corepoints'] & ref['corepoints'])
    spatial_union_count = det['n_corepoints'] + ref['n_corepoints'] - shared_corepoint_count
    spatial_iou = (
        shared_corepoint_count / spatial_union_count
        if spatial_union_count > 0
        else np.nan
    )

    overlapping_duration = max(
        0.0,
        min(det['end_time'], ref['end_time'])
        - max(det['start_time'], ref['start_time']),
    )
    temporal_union = det['duration'] + ref['duration'] - overlapping_duration
    temporal_iou = overlapping_duration / temporal_union if temporal_union > 0 else np.nan

    detected_st_support = det['n_corepoints'] * det['duration']
    reference_st_support = ref['n_corepoints'] * ref['duration']
    intersection_st_support = shared_corepoint_count * overlapping_duration
    union_st_support = detected_st_support + reference_st_support - intersection_st_support
    spatiotemporal_iou = (
        intersection_st_support / union_st_support
        if union_st_support > 0
        else np.nan
    )

    if np.isfinite(spatiotemporal_iou):
        if np.isfinite(spatial_iou) and spatiotemporal_iou > spatial_iou + _IOU_TOL:
            raise AssertionError("Spatiotemporal IoU exceeds spatial IoU")
        if np.isfinite(temporal_iou) and spatiotemporal_iou > temporal_iou + _IOU_TOL:
            raise AssertionError("Spatiotemporal IoU exceeds temporal IoU")

    return {
        'detected_corepoint_count': det['n_corepoints'],
        'reference_corepoint_count': ref['n_corepoints'],
        'shared_corepoint_count': shared_corepoint_count,
        'detected_start_time': det['start_time'],
        'detected_end_time': det['end_time'],
        'reference_start_time': ref['start_time'],
        'reference_end_time': ref['end_time'],
        'detected_duration': det['duration'],
        'reference_duration': ref['duration'],
        'overlapping_duration': overlapping_duration,
        'spatial_iou': spatial_iou,
        'temporal_iou': temporal_iou,
        'spatiotemporal_iou': spatiotemporal_iou,
        'detected_spatiotemporal_support': detected_st_support,
        'reference_spatiotemporal_support': reference_st_support,
        'intersection_spatiotemporal_support': intersection_st_support,
        'union_spatiotemporal_support': union_st_support,
    }


def _pairwise_metrics_table(method, detected_objects, reference_objects, det_time_days, ref_time_days):
    rows = []
    for detected_idx, det_obj in enumerate(detected_objects):
        for reference_idx, ref_obj in enumerate(reference_objects):
            row = {
                'method': method,
                'detected_object_index': detected_idx,
                'reference_object_index': reference_idx,
            }
            row.update(_pairwise_object_metrics(det_obj, ref_obj, det_time_days, ref_time_days))
            rows.append(row)
    return rows


def _spatiotemporal_iou_matrix(pair_rows, n_detected, n_reference):
    matrix = np.full((n_detected, n_reference), np.nan, dtype=float)
    for row in pair_rows:
        matrix[row['detected_object_index'], row['reference_object_index']] = row[
            'spatiotemporal_iou'
        ]
    return matrix


def _one_to_one_matches(spatiotemporal_iou_matrix, threshold):
    if 0 in spatiotemporal_iou_matrix.shape:
        return []

    cost = np.where(
        np.isfinite(spatiotemporal_iou_matrix),
        1.0 - spatiotemporal_iou_matrix,
        1e6,
    )
    detected_indices, reference_indices = linear_sum_assignment(cost)
    matches = []
    for detected_idx, reference_idx in zip(detected_indices, reference_indices):
        iou = spatiotemporal_iou_matrix[detected_idx, reference_idx]
        if np.isfinite(iou) and iou >= threshold:
            matches.append((int(detected_idx), int(reference_idx)))
    return matches


def _merge_intervals(intervals):
    merged = []
    for start_time, end_time in sorted(intervals):
        if not np.isfinite(start_time) or not np.isfinite(end_time):
            continue
        if end_time < start_time:
            raise ValueError(f"Invalid interval [{start_time}, {end_time}]")
        if not merged or start_time > merged[-1][1] + _IOU_TOL:
            merged.append([float(start_time), float(end_time)])
        else:
            merged[-1][1] = max(merged[-1][1], float(end_time))
    return [(start_time, end_time) for start_time, end_time in merged]


def _intervals_by_corepoint(objects_for_method, time_days):
    intervals = {}
    for obj in objects_for_method:
        props = _object_properties(obj, time_days)
        if props['duration'] == 0:
            continue
        interval = (props['start_time'], props['end_time'])
        for corepoint_idx in props['corepoints']:
            intervals.setdefault(corepoint_idx, []).append(interval)
    return {
        corepoint_idx: _merge_intervals(corepoint_intervals)
        for corepoint_idx, corepoint_intervals in intervals.items()
    }


def _total_interval_duration(intervals):
    return float(sum(end_time - start_time for start_time, end_time in intervals))


def _shared_interval_duration(left_intervals, right_intervals):
    total = 0.0
    left_idx = 0
    right_idx = 0
    while left_idx < len(left_intervals) and right_idx < len(right_intervals):
        left_start, left_end = left_intervals[left_idx]
        right_start, right_end = right_intervals[right_idx]
        total += max(0.0, min(left_end, right_end) - max(left_start, right_start))
        if left_end <= right_end:
            left_idx += 1
        else:
            right_idx += 1
    return total


def _global_support_metrics(detected_objects, reference_objects, det_time_days, ref_time_days):
    detected_by_corepoint = _intervals_by_corepoint(detected_objects, det_time_days)
    reference_by_corepoint = _intervals_by_corepoint(reference_objects, ref_time_days)
    corepoint_indices = set(detected_by_corepoint) | set(reference_by_corepoint)

    tp = 0.0
    detected_total = 0.0
    reference_total = 0.0
    for corepoint_idx in corepoint_indices:
        detected_intervals = detected_by_corepoint.get(corepoint_idx, [])
        reference_intervals = reference_by_corepoint.get(corepoint_idx, [])
        detected_total += _total_interval_duration(detected_intervals)
        reference_total += _total_interval_duration(reference_intervals)
        tp += _shared_interval_duration(detected_intervals, reference_intervals)

    fp = max(0.0, detected_total - tp)
    fn = max(0.0, reference_total - tp)
    precision = tp / (tp + fp) if tp + fp > 0 else np.nan
    recall = tp / (tp + fn) if tp + fn > 0 else np.nan
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else np.nan
    iou = tp / (tp + fp + fn) if tp + fp + fn > 0 else np.nan

    return {
        'global_tp': tp,
        'global_fp': fp,
        'global_fn': fn,
        'global_precision': precision,
        'global_recall': recall,
        'global_f1': f1,
        'global_iou': iou,
    }


def _trapezoid(values, times):
    if hasattr(np, 'trapezoid'):
        return float(np.trapezoid(values, times))
    return float(np.trapz(values, times))


def _mean_time_integrated_change(obj, distances, time_days):
    props = _object_properties(obj, time_days)
    epoch_slice = slice(props['start_epoch'], props['end_epoch'] + 1)
    object_times = time_days[epoch_slice] - time_days[props['start_epoch']]
    if len(object_times) < 2:
        return {
            'change_volume': np.nan,
            'valid_corepoint_count': 0,
            'valid_corepoint_percentage': 0.0,
        }

    integrated_changes = []
    for corepoint_idx in sorted(props['corepoints']):
        series = np.asarray(distances[corepoint_idx, epoch_slice], dtype=float)
        if not np.isfinite(series[0]):
            continue
        finite = np.isfinite(series)
        if finite.sum() < 2:
            continue
        change = np.abs(series[finite] - series[0])
        integrated_changes.append(_trapezoid(change, object_times[finite]))

    valid_count = len(integrated_changes)
    return {
        'change_volume': float(np.mean(integrated_changes)) if valid_count else np.nan,
        'valid_corepoint_count': valid_count,
        'valid_corepoint_percentage': 100.0 * valid_count / props['n_corepoints'],
    }


def _summary_stats(values, prefix):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return {
            f'mean_{prefix}': np.nan,
            f'median_{prefix}': np.nan,
            f'std_{prefix}': np.nan,
            f'min_{prefix}': np.nan,
            f'max_{prefix}': np.nan,
        }
    return {
        f'mean_{prefix}': float(np.mean(values)),
        f'median_{prefix}': float(np.median(values)),
        f'std_{prefix}': float(np.std(values)),
        f'min_{prefix}': float(np.min(values)),
        f'max_{prefix}': float(np.max(values)),
    }


def _harmonic_mean(precision, recall):
    return 2 * precision * recall / (precision + recall) if precision + recall > 0 else np.nan


def _evaluate_method(method, method_analysis, gt_analysis, timestamp_info):
    detected_objects = list(method_analysis.objects)
    reference_objects = list(gt_analysis.objects)
    detected_timestamps, detected_time_days = timestamp_info[method]
    reference_timestamps, reference_time_days = timestamp_info['GT']

    pair_rows = _pairwise_metrics_table(
        method,
        detected_objects,
        reference_objects,
        detected_time_days,
        reference_time_days,
    )
    pair_lookup = {
        (row['detected_object_index'], row['reference_object_index']): row
        for row in pair_rows
    }
    iou_matrix = _spatiotemporal_iou_matrix(
        pair_rows,
        len(detected_objects),
        len(reference_objects),
    )
    matched_pairs = _one_to_one_matches(iou_matrix, match_iou_threshold)

    matched_rows = []
    for detected_idx, reference_idx in matched_pairs:
        pair_row = pair_lookup[(detected_idx, reference_idx)]
        detected_change = _mean_time_integrated_change(
            detected_objects[detected_idx],
            method_analysis.distances,
            detected_time_days,
        )
        reference_change = _mean_time_integrated_change(
            reference_objects[reference_idx],
            gt_analysis.distances,
            reference_time_days,
        )
        change_difference = detected_change['change_volume'] - reference_change['change_volume']
        absolute_change_difference = abs(change_difference)

        matched_rows.append({
            'method': method,
            'detected_object_index': detected_idx,
            'reference_object_index': reference_idx,
            'spatial_iou': pair_row['spatial_iou'],
            'temporal_iou': pair_row['temporal_iou'],
            'spatiotemporal_iou': pair_row['spatiotemporal_iou'],
            'detected_corepoint_count': pair_row['detected_corepoint_count'],
            'reference_corepoint_count': pair_row['reference_corepoint_count'],
            'shared_corepoint_count': pair_row['shared_corepoint_count'],
            'detected_start_timestamp': detected_timestamps[int(detected_objects[detected_idx].start_epoch)],
            'detected_end_timestamp': detected_timestamps[int(detected_objects[detected_idx].end_epoch)],
            'reference_start_timestamp': reference_timestamps[int(reference_objects[reference_idx].start_epoch)],
            'reference_end_timestamp': reference_timestamps[int(reference_objects[reference_idx].end_epoch)],
            'detected_duration_days': pair_row['detected_duration'],
            'reference_duration_days': pair_row['reference_duration'],
            'overlapping_duration_days': pair_row['overlapping_duration'],
            'detected_change_volume': detected_change['change_volume'],
            'reference_change_volume': reference_change['change_volume'],
            'change_volume_difference': change_difference,
            'absolute_change_volume_difference': absolute_change_difference,
            'detected_valid_corepoint_count': detected_change['valid_corepoint_count'],
            'reference_valid_corepoint_count': reference_change['valid_corepoint_count'],
            'detected_valid_corepoint_percentage': detected_change['valid_corepoint_percentage'],
            'reference_valid_corepoint_percentage': reference_change['valid_corepoint_percentage'],
        })

    n_detected = len(detected_objects)
    n_reference = len(reference_objects)
    n_matched = len(matched_rows)
    object_precision = n_matched / n_detected if n_detected else np.nan
    object_recall = n_matched / n_reference if n_reference else np.nan
    object_f1 = _harmonic_mean(object_precision, object_recall)

    summary = {
        'method': method,
        'n_detected_objects': n_detected,
        'n_reference_objects': n_reference,
        'n_matched_pairs': n_matched,
        'n_unmatched_detected_objects': n_detected - n_matched,
        'n_unmatched_reference_objects': n_reference - n_matched,
        'object_precision': object_precision,
        'object_recall': object_recall,
        'object_f1': object_f1,
    }
    summary.update(_global_support_metrics(detected_objects, reference_objects, detected_time_days, reference_time_days))

    for column in ['spatial_iou', 'temporal_iou', 'spatiotemporal_iou']:
        summary.update(_summary_stats([row[column] for row in matched_rows], column))

    for column in ['change_volume_difference', 'absolute_change_volume_difference']:
        stats = _summary_stats([row[column] for row in matched_rows], column)
        summary[f'mean_{column}'] = stats[f'mean_{column}']
        summary[f'median_{column}'] = stats[f'median_{column}']

    return summary, matched_rows, pair_rows


class _FakeObject:
    def __init__(self, indices, start_epoch, end_epoch):
        self.indices = indices
        self.start_epoch = start_epoch
        self.end_epoch = end_epoch


def _assert_close(actual, expected, label):
    if not np.isclose(actual, expected, equal_nan=True):
        raise AssertionError(f"{label}: expected {expected}, got {actual}")


def _run_evaluation_self_tests():
    irregular_time_days = np.array([0.0, 1.0, 3.0, 6.0])

    metrics = _pairwise_object_metrics(
        _FakeObject([1, 2], 0, 2),
        _FakeObject([1, 2], 0, 2),
        irregular_time_days,
        irregular_time_days,
    )
    _assert_close(metrics['spatial_iou'], 1.0, 'identical spatial IoU')
    _assert_close(metrics['temporal_iou'], 1.0, 'identical temporal IoU')
    _assert_close(metrics['spatiotemporal_iou'], 1.0, 'identical spatiotemporal IoU')

    metrics = _pairwise_object_metrics(
        _FakeObject([1, 2], 0, 2),
        _FakeObject([1, 2], 1, 3),
        irregular_time_days,
        irregular_time_days,
    )
    _assert_close(metrics['spatial_iou'], 1.0, 'same spatial support')
    _assert_close(metrics['spatiotemporal_iou'], metrics['temporal_iou'], 'same spatial support ST IoU')

    metrics = _pairwise_object_metrics(
        _FakeObject([1, 2], 0, 2),
        _FakeObject([2, 3], 0, 2),
        irregular_time_days,
        irregular_time_days,
    )
    _assert_close(metrics['temporal_iou'], 1.0, 'same temporal interval')
    _assert_close(metrics['spatiotemporal_iou'], metrics['spatial_iou'], 'same temporal interval ST IoU')

    metrics = _pairwise_object_metrics(
        _FakeObject([1, 2], 0, 2),
        _FakeObject([2, 3], 1, 3),
        irregular_time_days,
        irregular_time_days,
    )
    if not (
        metrics['spatiotemporal_iou'] <= metrics['spatial_iou'] + _IOU_TOL
        and metrics['spatiotemporal_iou'] <= metrics['temporal_iou'] + _IOU_TOL
    ):
        raise AssertionError('partial overlap ST IoU bound failed')
    if np.isclose(
        metrics['spatiotemporal_iou'],
        metrics['spatial_iou'] * metrics['temporal_iou'],
    ):
        raise AssertionError('ST IoU should not be calculated as spatial_iou * temporal_iou')

    metrics = _pairwise_object_metrics(
        _FakeObject([1, 2], 0, 1),
        _FakeObject([1, 2], 2, 3),
        irregular_time_days,
        irregular_time_days,
    )
    _assert_close(metrics['temporal_iou'], 0.0, 'no temporal overlap')
    _assert_close(metrics['spatiotemporal_iou'], 0.0, 'no temporal overlap ST IoU')

    metrics = _pairwise_object_metrics(
        _FakeObject([1], 0, 2),
        _FakeObject([2], 0, 2),
        irregular_time_days,
        irregular_time_days,
    )
    _assert_close(metrics['spatial_iou'], 0.0, 'no spatial overlap')
    _assert_close(metrics['spatiotemporal_iou'], 0.0, 'no spatial overlap ST IoU')

    pair_rows = _pairwise_metrics_table(
        'test',
        [_FakeObject([1, 2], 0, 2), _FakeObject([1, 2], 1, 3)],
        [_FakeObject([1, 2], 0, 2)],
        irregular_time_days,
        irregular_time_days,
    )
    matrix = _spatiotemporal_iou_matrix(pair_rows, 2, 1)
    matches = _one_to_one_matches(matrix, 0.0)
    if len(matches) != 1 or matches[0] != (0, 0):
        raise AssertionError('one-to-one assignment competition failed')

    metrics = _pairwise_object_metrics(
        _FakeObject([1], 0, 1),
        _FakeObject([1], 0, 2),
        irregular_time_days,
        irregular_time_days,
    )
    _assert_close(metrics['temporal_iou'], 1.0 / 3.0, 'irregular timestamp temporal IoU')

    global_metrics = _global_support_metrics(
        [_FakeObject([1], 0, 2), _FakeObject([1], 1, 3)],
        [_FakeObject([1], 0, 3)],
        irregular_time_days,
        irregular_time_days,
    )
    _assert_close(global_metrics['global_tp'], 6.0, 'merged overlapping global TP')
    _assert_close(global_metrics['global_fp'], 0.0, 'merged overlapping global FP')
    _assert_close(global_metrics['global_fn'], 0.0, 'merged overlapping global FN')

    distances = np.array([
        [0.0, 1.0, np.nan],
        [np.nan, 2.0, 3.0],
        [0.0, np.nan, 3.0],
    ])
    change = _mean_time_integrated_change(
        _FakeObject([0, 1, 2], 0, 2),
        distances,
        irregular_time_days,
    )
    if change['valid_corepoint_count'] != 2:
        raise AssertionError('NaN handling should exclude only invalid corepoints')
    _assert_close(change['valid_corepoint_percentage'], 100.0 * 2 / 3, 'valid corepoint percentage')


_run_evaluation_self_tests()
print('Continuous 4D-OBC evaluation self-tests passed.')

_validate_common_corepoints(analyses)
timestamp_info = {name: _timestamps_for_analysis(st_analysis) for name, st_analysis in analyses.items()}
base_timestamps = timestamp_info['GT'][0]
for name, (timestamps_for_method, _) in timestamp_info.items():
    if len(timestamps_for_method) != len(base_timestamps) or any(
        ts != base_ts for ts, base_ts in zip(timestamps_for_method, base_timestamps)
    ):
        raise ValueError(f"{name} timestamps differ from GT timestamps")

print(
    f"Evaluation time axis: {len(base_timestamps)} acquisitions, "
    f"{timestamp_info['GT'][1][0]:.3f} to {timestamp_info['GT'][1][-1]:.3f} elapsed days"
)

gt_analysis = analyses['GT']
summary_rows = []
matched_object_rows = []
pairwise_metric_rows = []
for method in ['M3C2', 'TAM3C2']:
    summary, method_matches, method_pairs = _evaluate_method(
        method,
        analyses[method],
        gt_analysis,
        timestamp_info,
    )
    summary_rows.append(summary)
    matched_object_rows.extend(method_matches)
    pairwise_metric_rows.extend(method_pairs)

if pd is not None:
    summary_df = pd.DataFrame(summary_rows).set_index('method')
    matched_objects_df = pd.DataFrame(matched_object_rows)
    pairwise_metrics_df = pd.DataFrame(pairwise_metric_rows)

    summary_columns = [
        'n_detected_objects',
        'n_reference_objects',
        'n_matched_pairs',
        'n_unmatched_detected_objects',
        'n_unmatched_reference_objects',
        'object_precision',
        'object_recall',
        'object_f1',
        'global_tp',
        'global_fp',
        'global_fn',
        'global_precision',
        'global_recall',
        'global_f1',
        'global_iou',
        'mean_spatial_iou',
        'median_spatial_iou',
        'std_spatial_iou',
        'min_spatial_iou',
        'max_spatial_iou',
        'mean_temporal_iou',
        'median_temporal_iou',
        'std_temporal_iou',
        'min_temporal_iou',
        'max_temporal_iou',
        'mean_spatiotemporal_iou',
        'median_spatiotemporal_iou',
        'std_spatiotemporal_iou',
        'min_spatiotemporal_iou',
        'max_spatiotemporal_iou',
        'mean_change_volume_difference',
        'median_change_volume_difference',
        'mean_absolute_change_volume_difference',
        'median_absolute_change_volume_difference',
    ]
    display(summary_df[summary_columns])

    if len(matched_objects_df) > 0:
        display(
            matched_objects_df.sort_values(
                ['method', 'spatiotemporal_iou'],
                ascending=[True, False],
            )
        )
    else:
        print('No successful one-to-one object matches at the configured threshold.')
else:
    summary_df = summary_rows
    matched_objects_df = matched_object_rows
    pairwise_metrics_df = pairwise_metric_rows
    print('Summary:')
    for row in summary_rows:
        print(row)
    print('Successful one-to-one matches:')
    for row in matched_object_rows:
        print(row)


In [ ]:
# Visual summary of the continuous 4D-OBC comparison.

if pd is not None:
    global_metrics_to_plot = ['global_precision', 'global_recall', 'global_f1', 'global_iou']
    ax = (
        summary_df[global_metrics_to_plot]
        .rename(columns={
            'global_precision': 'Precision',
            'global_recall': 'Recall',
            'global_f1': 'F1',
            'global_iou': 'IoU',
        })
        .plot(kind='bar', figsize=(9, 4), ylim=(0, 1), rot=0)
    )
    ax.set_ylabel('Score')
    ax.set_title('Global continuous-support agreement against GT')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    object_metrics_to_plot = ['object_precision', 'object_recall', 'object_f1']
    ax = (
        summary_df[object_metrics_to_plot]
        .rename(columns={
            'object_precision': 'Object precision',
            'object_recall': 'Object recall',
            'object_f1': 'Object F1',
        })
        .plot(kind='bar', figsize=(9, 4), ylim=(0, 1), rot=0)
    )
    ax.set_ylabel('Score')
    ax.set_title('One-to-one object detection metrics')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    if len(matched_objects_df) > 0 and 'method' in matched_objects_df.columns:
        fig, ax = plt.subplots(figsize=(9, 4))
        for method, group in matched_objects_df.groupby('method'):
            vals = np.sort(group['spatiotemporal_iou'].to_numpy())[::-1]
            ax.plot(np.arange(1, len(vals) + 1), vals, marker='o', ms=3, lw=1, label=method)
        ax.axhline(match_iou_threshold, color='black', ls='--', lw=1, label=f'match threshold={match_iou_threshold}')
        ax.set_xlabel('Successful one-to-one match rank by spatiotemporal IoU')
        ax.set_ylabel('Spatiotemporal IoU')
        ax.set_title('Matched-object quality')
        ax.grid(alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.show()
    else:
        print('No successful one-to-one object matches to plot.')
else:
    print('Install pandas to enable table-based plotting in this cell.')

表示被 Hungarian algorithm 选中的一对一对象对，且iou>=0.8  
下面是最终reference 一对一 匹配结果 （最多到ref的数量 71）

In [ ]:
from scipy.optimize import linear_sum_assignment


# ================================================================
# All Hungarian one-to-one assignments without IoU filtering
# ================================================================

all_assignment_rows = []
assignment_summary_rows = []


for method in METHODS:
    method_pairs = pairwise_topology_df.loc[
        pairwise_topology_df["method"]
        == method
    ].copy()

    n_detected = len(
        analyses[method].objects
    )

    n_reference = len(
        analyses["GT"].objects
    )


    # Rows: detected objects
    # Columns: reference objects
    st_iou_matrix = np.full(
        (
            n_detected,
            n_reference,
        ),
        np.nan,
        dtype=float,
    )


    for pair in method_pairs.itertuples(
        index=False
    ):
        detected_idx = int(
            pair.detected_object_index
        )

        reference_idx = int(
            pair.reference_object_index
        )

        st_iou_matrix[
            detected_idx,
            reference_idx,
        ] = float(
            pair.spatiotemporal_iou
        )


    cost_matrix = np.where(
        np.isfinite(st_iou_matrix),
        1.0 - st_iou_matrix,
        1e6,
    )


    assigned_detected_indices, (
        assigned_reference_indices
    ) = linear_sum_assignment(
        cost_matrix
    )


    method_assignment_rows = []


    pair_lookup = (
        method_pairs
        .set_index(
            [
                "detected_object_index",
                "reference_object_index",
            ]
        )
    )


    for detected_idx, reference_idx in zip(
        assigned_detected_indices,
        assigned_reference_indices,
    ):
        detected_idx = int(detected_idx)
        reference_idx = int(reference_idx)

        pair_row = pair_lookup.loc[
            (
                detected_idx,
                reference_idx,
            )
        ]

        assignment_row = {
            "method": method,
            "detected_object_index": (
                detected_idx
            ),
            "reference_object_index": (
                reference_idx
            ),
            "spatial_iou": float(
                pair_row["spatial_iou"]
            ),
            "temporal_iou": float(
                pair_row["temporal_iou"]
            ),
            "spatiotemporal_iou": float(
                pair_row[
                    "spatiotemporal_iou"
                ]
            ),
        }

        method_assignment_rows.append(
            assignment_row
        )


    method_assignment_rows = sorted(
        method_assignment_rows,
        key=lambda row: (
            row["spatiotemporal_iou"]
        ),
        reverse=True,
    )


    for rank, row in enumerate(
        method_assignment_rows,
        start=1,
    ):
        row["assignment_rank"] = rank
        all_assignment_rows.append(row)


    n_assignments = len(
        method_assignment_rows
    )

    n_above_threshold = sum(
        row["spatiotemporal_iou"]
        >= MATCH_THRESHOLD
        for row in method_assignment_rows
    )

    n_below_threshold = sum(
        (
            row["spatiotemporal_iou"]
            < MATCH_THRESHOLD
            and row[
                "spatiotemporal_iou"
            ] > IOU_EPS
        )
        for row in method_assignment_rows
    )

    n_zero_assignments = sum(
        row["spatiotemporal_iou"]
        <= IOU_EPS
        for row in method_assignment_rows
    )


    assignment_summary_rows.append(
        {
            "method": method,
            "n_detected_objects": (
                n_detected
            ),
            "n_reference_objects": (
                n_reference
            ),
            "n_hungarian_assignments": (
                n_assignments
            ),
            "n_assignments_iou_ge_0.8": (
                n_above_threshold
            ),
            "n_assignments_0_lt_iou_lt_0.8": (
                n_below_threshold
            ),
            "n_assignments_iou_zero": (
                n_zero_assignments
            ),
            "n_unassigned_detected_objects": (
                n_detected - n_assignments
            ),
            "n_unassigned_reference_objects": (
                n_reference - n_assignments
            ),
        }
    )


all_hungarian_assignments_df = (
    pd.DataFrame(all_assignment_rows)
)

all_hungarian_summary_df = (
    pd.DataFrame(assignment_summary_rows)
    .set_index("method")
)


# ================================================================
# Plot every Hungarian assignment
# ================================================================

fig, ax = plt.subplots(
    figsize=(10, 6),
    dpi=130,
    constrained_layout=True,
)


method_colors = {
    "M3C2": "#4C78A8",
    "TAM3C2": "#F58518",
}


for method in METHODS:
    method_assignments = (
        all_hungarian_assignments_df.loc[
            all_hungarian_assignments_df[
                "method"
            ] == method
        ]
        .sort_values(
            "assignment_rank"
        )
    )

    ax.plot(
        method_assignments[
            "assignment_rank"
        ],
        method_assignments[
            "spatiotemporal_iou"
        ],
        marker="o",
        markersize=4,
        linewidth=1.5,
        color=method_colors[method],
        label=method,
    )


ax.axhline(
    MATCH_THRESHOLD,
    color="black",
    linestyle="--",
    linewidth=1.1,
    label=(
        f"Match threshold = "
        f"{MATCH_THRESHOLD:.2f}"
    ),
)

ax.set_xlabel(
    "Hungarian assignment rank "
    "by spatiotemporal IoU"
)

ax.set_ylabel(
    "Spatiotemporal IoU"
)

ax.set_title(
    "All Hungarian one-to-one assignments "
    "without IoU filtering"
)

ax.set_ylim(
    -0.02,
    1.03,
)

ax.grid(
    alpha=0.25,
)

ax.legend(
    frameon=False,
)

plt.show()


print(
    "All Hungarian assignment summary"
)

display(all_hungarian_summary_df)


print(
    "All Hungarian assignment pairs"
)

display(
    all_hungarian_assignments_df.sort_values(
        [
            "method",
            "assignment_rank",
        ]
    )
)

# old comparison

In [ ]:
# The archives were loaded in Section 2. Keep the shared dictionary interface
# used by the evaluation code below.
analyses = {
    'GT': ref_analysis,
    'M3C2': m3c2_analysis,
    'TAM3C2': analysis,
}

for name, st_analysis in analyses.items():
    objects_for_method = st_analysis.objects
    if objects_for_method is None:
        raise RuntimeError(
            f"{name} analysis has no stored 4D-OBC objects. "
            "Run RegionGrowingAlgorithm in the producer notebook first."
        )
    print(
        f"{name}: {len(objects_for_method)} objects, "
        f"{st_analysis.corepoints.cloud.shape[0]:,} corepoints, "
        f"distance shape={st_analysis.distances.shape}, "
        f"path={st_analysis.filename}"
    )


In [ ]:
try:
    import pandas as pd
except ImportError:
    pd = None


def _corepoint_index_mapper(source_analysis, target_analysis):
    source_cp = np.asarray(source_analysis.corepoints.cloud)
    target_cp = np.asarray(target_analysis.corepoints.cloud)

    if source_cp.shape != target_cp.shape or not np.allclose(source_cp, target_cp):
        raise ValueError(
            "All compared archives must use the same corepoint array and ordering."
        )
    return np.arange(len(source_cp)), np.ones(len(source_cp), dtype=bool)


def _object_corepoint_set(obj, index_map=None, valid_source=None):
    idx = np.asarray(obj.indices, dtype=int)
    if index_map is None:
        return set(idx.tolist())

    keep = valid_source[idx]
    return set(index_map[idx[keep]].tolist())


def _object_epoch_set(obj):
    return set(range(int(obj.start_epoch), int(obj.end_epoch) + 1))


def _object_token_set(obj, index_map=None, valid_source=None):
    cp_set = _object_corepoint_set(obj, index_map=index_map, valid_source=valid_source)
    epoch_set = _object_epoch_set(obj)
    return {(cp_idx, epoch_idx) for cp_idx in cp_set for epoch_idx in epoch_set}


def _analysis_token_set(objects_for_method, index_map=None, valid_source=None):
    tokens = set()
    for obj in objects_for_method:
        tokens |= _object_token_set(obj, index_map=index_map, valid_source=valid_source)
    return tokens


def _set_metrics(pred_tokens, gt_tokens):
    tp = len(pred_tokens & gt_tokens)
    fp = len(pred_tokens - gt_tokens)
    fn = len(gt_tokens - pred_tokens)
    union = len(pred_tokens | gt_tokens)
    precision = tp / (tp + fp) if tp + fp else np.nan
    recall = tp / (tp + fn) if tp + fn else np.nan
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else np.nan
    iou = tp / union if union else np.nan
    return {
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'iou': iou,
    }


def _iou(a, b):
    union = len(a | b)
    return len(a & b) / union if union else np.nan


def _match_objects(pred_objects, gt_objects, index_map=None, valid_source=None):
    gt_cp_sets = [_object_corepoint_set(obj) for obj in gt_objects]
    gt_epoch_sets = [_object_epoch_set(obj) for obj in gt_objects]
    gt_token_sets = [_object_token_set(obj) for obj in gt_objects]

    rows = []
    for pred_idx, pred_obj in enumerate(pred_objects):
        pred_cp = _object_corepoint_set(pred_obj, index_map=index_map, valid_source=valid_source)
        pred_epochs = _object_epoch_set(pred_obj)
        pred_tokens = _object_token_set(pred_obj, index_map=index_map, valid_source=valid_source)

        best = None
        for gt_idx, (gt_cp, gt_epochs, gt_tokens) in enumerate(zip(gt_cp_sets, gt_epoch_sets, gt_token_sets)):
            st_iou = _iou(pred_tokens, gt_tokens)
            row = {
                'pred_object': pred_idx,
                'gt_object': gt_idx,
                'spatial_iou': _iou(pred_cp, gt_cp),
                'temporal_iou': _iou(pred_epochs, gt_epochs),
                'spacetime_iou': st_iou,
                'pred_corepoints': len(pred_cp),
                'gt_corepoints': len(gt_cp),
                'pred_epochs': len(pred_epochs),
                'gt_epochs': len(gt_epochs),
            }
            if best is None or st_iou > best['spacetime_iou']:
                best = row

        if best is not None:
            best['matched'] = best['spacetime_iou'] >= match_iou_threshold
            rows.append(best)

    return rows


gt_analysis = analyses['GT']
gt_objects = list(gt_analysis.objects)
gt_tokens = _analysis_token_set(gt_objects)

summary_rows = []
match_rows = []
for method in ['M3C2', 'TAM3C2']:
    method_analysis = analyses[method]
    method_objects = list(method_analysis.objects)
    index_map, valid_source = _corepoint_index_mapper(method_analysis, gt_analysis)

    method_tokens = _analysis_token_set(
        method_objects,
        index_map=index_map,
        valid_source=valid_source,
    )
    metrics = _set_metrics(method_tokens, gt_tokens)
    metrics.update({
        'method': method,
        'n_objects': len(method_objects),
        'n_gt_objects': len(gt_objects),
        'matched_objects': 0,
    })

    rows = _match_objects(
        method_objects,
        gt_objects,
        index_map=index_map,
        valid_source=valid_source,
    )
    for row in rows:
        row['method'] = method
    metrics['matched_objects'] = sum(row['matched'] for row in rows)

    summary_rows.append(metrics)
    match_rows.extend(rows)

if pd is not None:
    summary_df = pd.DataFrame(summary_rows).set_index('method')
    matches_df = pd.DataFrame(match_rows)
    display(summary_df[['n_objects', 'n_gt_objects', 'matched_objects', 'precision', 'recall', 'f1', 'iou', 'tp', 'fp', 'fn']])
    if len(matches_df) > 0:
        display(matches_df.sort_values(['method', 'spacetime_iou'], ascending=[True, False]))
    else:
        print('No object-level matches to display because no predicted objects or no GT objects were available.')
else:
    summary_df = summary_rows
    matches_df = match_rows
    print('Summary:')
    for row in summary_rows:
        print(row)
    print('Top matches:')
    for row in sorted(match_rows, key=lambda r: r['spacetime_iou'], reverse=True)[:20]:
        print(row)


In [ ]:

if pd is not None:
    metrics_to_plot = ['precision', 'recall', 'f1', 'iou']
    ax = summary_df[metrics_to_plot].plot(kind='bar', figsize=(9, 4), ylim=(0, 1), rot=0)
    ax.set_ylabel('Score')
    ax.set_title('Global 4D-OBC agreement against GT')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    if len(matches_df) > 0 and 'method' in matches_df.columns:
        fig, ax = plt.subplots(figsize=(9, 4))
        for method, group in matches_df.groupby('method'):
            vals = np.sort(group['spacetime_iou'].to_numpy())[::-1]
            ax.plot(np.arange(1, len(vals) + 1), vals, marker='o', ms=3, lw=1, label=method)
        ax.axhline(match_iou_threshold, color='black', ls='--', lw=1, label=f'match threshold={match_iou_threshold}')
        ax.set_xlabel('Predicted object rank by best GT IoU')
        ax.set_ylabel('Best GT spacetime IoU')
        ax.set_title('Object-level best-match quality')
        ax.grid(alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.show()
    else:
        print('No object-level matches to plot.')
else:
    print('Install pandas to enable table-based plotting in this cell.')

# individual object investigation

In [ ]:
# Investigation: plot all 4D-OBC object delineations for GT, M3C2, and TAM3C2.
# Object delineation is shown as the convex hull of member corepoints.

from scipy.spatial import ConvexHull, QhullError
from matplotlib.patches import Polygon
from matplotlib.lines import Line2D

def _objects_for_analysis(st_analysis):
    objects_for_method = st_analysis.objects
    if objects_for_method is None:
        raise RuntimeError(f"{st_analysis.filename} has no stored 4D-OBC objects.")
    return list(objects_for_method)


def _object_indices(obj):
    return np.asarray(obj.indices, dtype=int)


def _object_xy(st_analysis, obj):
    corepoints_for_method = np.asarray(st_analysis.corepoints.cloud)
    return corepoints_for_method[_object_indices(obj), :2]


def _draw_object_outline(
    ax,
    xy,
    color,
    label=None,
    linewidth=1.4,
    point_size=5,
    point_alpha=0.25,
    fill_alpha=0.06,
):
    xy = np.asarray(xy, dtype=float)
    if len(xy) == 0:
        return None

    xy_unique = np.unique(xy, axis=0)
    ax.scatter(
        xy[:, 0],
        xy[:, 1],
        s=point_size,
        color=color,
        alpha=point_alpha,
        linewidths=0,
    )

    if len(xy_unique) >= 3:
        try:
            hull = ConvexHull(xy_unique)
            hull_xy = xy_unique[hull.vertices]
            patch = Polygon(
                hull_xy,
                closed=True,
                facecolor=color,
                edgecolor=color,
                linewidth=linewidth,
                alpha=fill_alpha,
            )
            ax.add_patch(patch)
            ax.plot(
                np.r_[hull_xy[:, 0], hull_xy[0, 0]],
                np.r_[hull_xy[:, 1], hull_xy[0, 1]],
                color=color,
                linewidth=linewidth,
                alpha=0.95,
                label=label,
            )
            return np.mean(hull_xy, axis=0)
        except QhullError:
            pass

    if len(xy_unique) >= 2:
        order = np.lexsort((xy_unique[:, 1], xy_unique[:, 0]))
        xy_line = xy_unique[order]
        ax.plot(
            xy_line[:, 0],
            xy_line[:, 1],
            color=color,
            linewidth=linewidth,
            alpha=0.95,
            label=label,
        )
    else:
        ax.scatter(
            xy_unique[:, 0],
            xy_unique[:, 1],
            s=30,
            color=color,
            marker="o",
            label=label,
        )

    return np.mean(xy_unique, axis=0)


def plot_all_object_delineations(
    analyses,
    method_names=("GT", "M3C2", "TAM3C2"),
    label_object_ids=True,
):
    fig, axs = plt.subplots(
        1,
        len(method_names),
        figsize=(6.2 * len(method_names), 6),
        constrained_layout=True,
    )
    if len(method_names) == 1:
        axs = [axs]

    for ax, name in zip(axs, method_names):
        st_analysis = analyses[name]
        corepoints_for_method = np.asarray(st_analysis.corepoints.cloud)
        objects_for_method = _objects_for_analysis(st_analysis)

        ax.scatter(
            corepoints_for_method[:, 0],
            corepoints_for_method[:, 1],
            s=0.3,
            color="0.88",
            linewidths=0,
            label="corepoints",
        )

        cmap = plt.colormaps.get_cmap("tab20")
        for object_id, obj in enumerate(objects_for_method):
            color = cmap(object_id % 20)
            xy = _object_xy(st_analysis, obj)
            centroid = _draw_object_outline(ax, xy, color=color)

            if label_object_ids and centroid is not None:
                ax.text(
                    centroid[0],
                    centroid[1],
                    str(object_id),
                    fontsize=8,
                    ha="center",
                    va="center",
                    color="black",
                    bbox=dict(facecolor="white", edgecolor="none", alpha=0.65, pad=1.0),
                )

        ax.set_title(f"{name}: all 4D-OBC object delineations\nn={len(objects_for_method)}")
        ax.set_xlabel("X [m]")
        ax.set_ylabel("Y [m]")
        ax.set_aspect("equal", adjustable="box")
        ax.grid(alpha=0.2)

    plt.show()


plot_all_object_delineations(analyses)

In [ ]:
reference_object_id = 0  

In [ ]:
# Select one GT/reference object and find its best M3C2/TAM3C2 matches.

try:
    import pandas as pd
except ImportError:
    pd = None

use_smoothed_time_series = False
METHODS = ("GT", "M3C2", "TAM3C2")
COLORS = {"GT": "black", "M3C2": "tab:blue", "TAM3C2": "tab:orange"}

print("Object diagnostics use raw st_analysis.distances, not smoothed_distances.")


def _timestamp_info_for(name):
    if "timestamp_info" in globals() and name in timestamp_info:
        return timestamp_info[name]
    return _timestamps_for_analysis(analyses[name])


def _analysis_object(name, object_id):
    return _objects_for_analysis(analyses[name])[object_id]


def _single_object_change_summary(name, object_id):
    _, time_days = _timestamp_info_for(name)
    return _mean_time_integrated_change(
        _analysis_object(name, object_id),
        np.asarray(analyses[name].distances, dtype=float),
        time_days,
    )


def _safe_percent_ratio(numerator, denominator):
    if not np.isfinite(numerator) or not np.isfinite(denominator) or denominator == 0:
        return np.nan
    return 100.0 * numerator / denominator


def _best_detected_match_for_reference(reference_object_id, method):
    gt_objects = _objects_for_analysis(analyses["GT"])
    detected_objects = _objects_for_analysis(analyses[method])

    if not 0 <= reference_object_id < len(gt_objects):
        raise IndexError(
            f"reference_object_id={reference_object_id} outside GT object range "
            f"[0, {len(gt_objects) - 1}]"
        )

    _, ref_time_days = _timestamp_info_for("GT")
    _, det_time_days = _timestamp_info_for(method)

    rows = []
    for detected_object_id, det_obj in enumerate(detected_objects):
        row = _pairwise_object_metrics(
            det_obj,
            gt_objects[reference_object_id],
            det_time_days,
            ref_time_days,
        )
        row.update(
            method=method,
            reference_object_id=reference_object_id,
            detected_object_id=detected_object_id,
        )
        rows.append(row)

    finite_rows = [row for row in rows if np.isfinite(row["spatiotemporal_iou"])]
    return (max(finite_rows, key=lambda row: row["spatiotemporal_iou"]) if finite_rows else None), rows


def build_single_reference_match_table(reference_object_id):
    gt_change = _single_object_change_summary("GT", reference_object_id)
    reference_change_volume = gt_change["change_volume"]

    rows = []
    best_matches = {}

    for method in ("M3C2", "TAM3C2"):
        best, _ = _best_detected_match_for_reference(reference_object_id, method)
        best_matches[method] = best

        if best is None:
            rows.append(
                dict(
                    method=method,
                    reference_object_id=reference_object_id,
                    best_detected_object_id=None,
                    spatial_iou_pct=np.nan,
                    temporal_iou_pct=np.nan,
                    spatiotemporal_iou_pct=np.nan,
                    reference_change_volume=reference_change_volume,
                    detected_change_volume=np.nan,
                    detected_reference_change_pct=np.nan,
                    change_difference=np.nan,
                    change_difference_pct_of_ref=np.nan,
                )
            )
            continue

        detected_object_id = int(best["detected_object_id"])
        det_change = _single_object_change_summary(method, detected_object_id)
        detected_change_volume = det_change["change_volume"]
        change_difference = detected_change_volume - reference_change_volume

        rows.append(
            dict(
                method=method,
                reference_object_id=reference_object_id,
                best_detected_object_id=detected_object_id,
                spatial_iou_pct=100.0 * best["spatial_iou"],
                temporal_iou_pct=100.0 * best["temporal_iou"],
                spatiotemporal_iou_pct=100.0 * best["spatiotemporal_iou"],
                reference_change_volume=reference_change_volume,
                detected_change_volume=detected_change_volume,
                detected_reference_change_pct=_safe_percent_ratio(
                    detected_change_volume,
                    reference_change_volume,
                ),
                change_difference=change_difference,
                change_difference_pct_of_ref=_safe_percent_ratio(
                    change_difference,
                    reference_change_volume,
                ),
                shared_corepoints=best["shared_corepoint_count"],
                reference_corepoints=best["reference_corepoint_count"],
                detected_corepoints=best["detected_corepoint_count"],
                overlapping_duration_days=best["overlapping_duration"],
                reference_duration_days=best["reference_duration"],
                detected_duration_days=best["detected_duration"],
            )
        )

    table = pd.DataFrame(rows) if pd is not None else rows
    return table, best_matches


def _object_interval_record(name, object_id):
    timestamps, time_days = _timestamp_info_for(name)
    obj = _analysis_object(name, object_id)
    props = _object_properties(obj, time_days)

    return dict(
        name=name,
        object_id=object_id,
        xy=_object_xy(analyses[name], obj),
        color=COLORS[name],
        interval=dict(
            start_epoch=props["start_epoch"],
            end_epoch=props["end_epoch"],
            start_timestamp=timestamps[props["start_epoch"]],
            end_timestamp=timestamps[props["end_epoch"]],
            start_time_days=props["start_time"],
            end_time_days=props["end_time"],
            duration_days=props["duration"],
        ),
    )


single_match_table, best_matches = build_single_reference_match_table(reference_object_id)

selected_object_ids = {"GT": reference_object_id}
selected_object_ids.update(
    {
        method: None if best_matches[method] is None else int(best_matches[method]["detected_object_id"])
        for method in ("M3C2", "TAM3C2")
    }
)

selected_extent_records = [
    _object_interval_record(name, object_id)
    for name, object_id in selected_object_ids.items()
    if object_id is not None
]

print(f"Selected GT/reference object id: {reference_object_id}")
for method, best in best_matches.items():
    if best is None:
        print(f"{method}: no finite best match")
    else:
        print(
            f"{method}: best object id={int(best['detected_object_id'])}, "
            f"ST-IoU={100.0 * best['spatiotemporal_iou']:.2f}%, "
            f"spatial IoU={100.0 * best['spatial_iou']:.2f}%, "
            f"temporal IoU={100.0 * best['temporal_iou']:.2f}%"
        )

if pd is not None:
    display(single_match_table)
else:
    for row in single_match_table:
        print(row)

In [ ]:
# Shared plotting helpers.
import matplotlib.dates as mdates
from matplotlib.patches import Polygon
from matplotlib.ticker import MaxNLocator
from scipy.spatial import ConvexHull, QhullError

# Shared publication style. Font sizes are specified in points, while the
# panel geometry is specified in inches. The standalone and combined changemap
# axes therefore have exactly the same physical size.
OBJECT_FIG_DPI = 120  # notebook display only; use 300 dpi for PNG export

OBJECT_FONT_SIZE = 14
OBJECT_TITLE_SIZE = 14
OBJECT_TICK_SIZE = 14
OBJECT_LEGEND_SIZE = 14
OBJECT_ANNOTATION_SIZE = 14
OBJECT_COLORBAR_SIZE = 12
OBJECT_COLORBAR_TICK_SIZE = 12

# Fixed physical geometry for every changemap panel.
OBJECT_PANEL_HEIGHT = 4.80
OBJECT_MAP_WIDTH = 4.80
OBJECT_TIME_SERIES_WIDTH = 9.60
OBJECT_LEFT_MARGIN = 0.95
OBJECT_RIGHT_MARGIN = 0.20
OBJECT_BOTTOM_MARGIN = 0.90
OBJECT_TOP_MARGIN = 0.50
OBJECT_PANEL_GAP = 0.70
OBJECT_COLORBAR_GAP = 0.16
OBJECT_COLORBAR_WIDTH = 0.25

OBJECT_COMMON_FIGURE_HEIGHT = (
    OBJECT_BOTTOM_MARGIN + OBJECT_PANEL_HEIGHT + OBJECT_TOP_MARGIN
)
OBJECT_SINGLE_CHANGEMAP_FIGSIZE = (
    OBJECT_LEFT_MARGIN
    + OBJECT_MAP_WIDTH
    + OBJECT_COLORBAR_GAP
    + OBJECT_COLORBAR_WIDTH
    + OBJECT_RIGHT_MARGIN,
    OBJECT_COMMON_FIGURE_HEIGHT,
)
OBJECT_TIME_SERIES_MAP_FIGSIZE = (
    OBJECT_LEFT_MARGIN
    + OBJECT_TIME_SERIES_WIDTH
    + OBJECT_PANEL_GAP
    + OBJECT_MAP_WIDTH
    + OBJECT_COLORBAR_GAP
    + OBJECT_COLORBAR_WIDTH
    + OBJECT_RIGHT_MARGIN,
    OBJECT_COMMON_FIGURE_HEIGHT,
)
OBJECT_TEMPORAL_FIGSIZE = (OBJECT_TIME_SERIES_MAP_FIGSIZE[0], 5.4)


def _axes_box(fig_width, fig_height, left, bottom, width, height):
    """Convert an axes rectangle from inches to figure fractions."""
    return [
        left / fig_width,
        bottom / fig_height,
        width / fig_width,
        height / fig_height,
    ]


def _new_single_changemap_figure():
    fig_width, fig_height = OBJECT_SINGLE_CHANGEMAP_FIGSIZE
    fig = plt.figure(
        figsize=OBJECT_SINGLE_CHANGEMAP_FIGSIZE,
        dpi=OBJECT_FIG_DPI,
    )
    ax_map = fig.add_axes(
        _axes_box(
            fig_width,
            fig_height,
            OBJECT_LEFT_MARGIN,
            OBJECT_BOTTOM_MARGIN,
            OBJECT_MAP_WIDTH,
            OBJECT_PANEL_HEIGHT,
        )
    )
    colorbar_left = (
        OBJECT_LEFT_MARGIN + OBJECT_MAP_WIDTH + OBJECT_COLORBAR_GAP
    )
    ax_colorbar = fig.add_axes(
        _axes_box(
            fig_width,
            fig_height,
            colorbar_left,
            OBJECT_BOTTOM_MARGIN,
            OBJECT_COLORBAR_WIDTH,
            OBJECT_PANEL_HEIGHT,
        )
    )
    return fig, ax_map, ax_colorbar


def _new_time_series_changemap_figure():
    fig_width, fig_height = OBJECT_TIME_SERIES_MAP_FIGSIZE
    fig = plt.figure(
        figsize=OBJECT_TIME_SERIES_MAP_FIGSIZE,
        dpi=OBJECT_FIG_DPI,
    )
    ax_time_series = fig.add_axes(
        _axes_box(
            fig_width,
            fig_height,
            OBJECT_LEFT_MARGIN,
            OBJECT_BOTTOM_MARGIN,
            OBJECT_TIME_SERIES_WIDTH,
            OBJECT_PANEL_HEIGHT,
        )
    )
    map_left = (
        OBJECT_LEFT_MARGIN + OBJECT_TIME_SERIES_WIDTH + OBJECT_PANEL_GAP
    )
    ax_map = fig.add_axes(
        _axes_box(
            fig_width,
            fig_height,
            map_left,
            OBJECT_BOTTOM_MARGIN,
            OBJECT_MAP_WIDTH,
            OBJECT_PANEL_HEIGHT,
        )
    )
    colorbar_left = map_left + OBJECT_MAP_WIDTH + OBJECT_COLORBAR_GAP
    ax_colorbar = fig.add_axes(
        _axes_box(
            fig_width,
            fig_height,
            colorbar_left,
            OBJECT_BOTTOM_MARGIN,
            OBJECT_COLORBAR_WIDTH,
            OBJECT_PANEL_HEIGHT,
        )
    )
    return fig, ax_time_series, ax_map, ax_colorbar

def _apply_object_plot_font_sizes(fig):
    for ax in fig.axes:
        is_colorbar = bool(getattr(ax, "_is_object_colorbar", False))

        ax.title.set_fontsize(OBJECT_TITLE_SIZE if not is_colorbar else OBJECT_COLORBAR_SIZE)
        ax.xaxis.label.set_fontsize(OBJECT_FONT_SIZE if not is_colorbar else OBJECT_COLORBAR_SIZE)
        ax.yaxis.label.set_fontsize(OBJECT_FONT_SIZE if not is_colorbar else OBJECT_COLORBAR_SIZE)
        ax.tick_params(
            axis="both",
            which="major",
            labelsize=OBJECT_COLORBAR_TICK_SIZE if is_colorbar else OBJECT_TICK_SIZE,
        )
        ax.tick_params(
            axis="both",
            which="minor",
            labelsize=OBJECT_COLORBAR_TICK_SIZE if is_colorbar else OBJECT_TICK_SIZE,
        )

        for text in ax.texts:
            text.set_fontsize(OBJECT_ANNOTATION_SIZE)

        legend = ax.get_legend()
        if legend is not None:
            for text in legend.get_texts():
                text.set_fontsize(OBJECT_LEGEND_SIZE)
            legend.get_title().set_fontsize(OBJECT_LEGEND_SIZE)

def _draw_object_extent(
    ax,
    xy,
    color,
    label,
    linewidth=2.2,
    point_size=10,
    point_alpha=0.55,
    fill_alpha=0.05,
):
    xy = np.asarray(xy, dtype=float)
    if len(xy) == 0:
        return

    xy_unique = np.unique(xy, axis=0)

    ax.scatter(
        xy[:, 0],
        xy[:, 1],
        s=point_size,
        color=color,
        alpha=point_alpha,
        linewidths=0,
        zorder=4,
    )

    if len(xy_unique) >= 3:
        try:
            hull = ConvexHull(xy_unique)
            hull_xy = xy_unique[hull.vertices]

            ax.add_patch(
                Polygon(
                    hull_xy,
                    closed=True,
                    facecolor=color,
                    edgecolor=color,
                    linewidth=linewidth,
                    alpha=fill_alpha,
                    zorder=5,
                )
            )
            ax.plot(
                np.r_[hull_xy[:, 0], hull_xy[0, 0]],
                np.r_[hull_xy[:, 1], hull_xy[0, 1]],
                color=color,
                linewidth=linewidth,
                label=label,
                zorder=6,
            )
            return
        except QhullError:
            pass

    if len(xy_unique) >= 2:
        order = np.lexsort((xy_unique[:, 1], xy_unique[:, 0]))
        xy_line = xy_unique[order]
        ax.plot(
            xy_line[:, 0],
            xy_line[:, 1],
            color=color,
            linewidth=linewidth,
            label=label,
            zorder=6,
        )
    else:
        ax.scatter(
            xy_unique[:, 0],
            xy_unique[:, 1],
            s=40,
            color=color,
            label=label,
            zorder=6,
        )

def _raw_changemap(record):
    name = record["name"]
    st_analysis = analyses[name]

    cloud = np.asarray(st_analysis.corepoints.cloud)
    distances = np.asarray(st_analysis.distances, dtype=float)

    start_epoch = int(record["interval"]["start_epoch"])
    end_epoch = int(record["interval"]["end_epoch"])
    magnitudes = distances[:, end_epoch] - distances[:, start_epoch]

    return cloud, magnitudes

def _scene_limits(ax, cloud, pad_ratio=0.02):
    xy = np.asarray(cloud[:, :2], dtype=float)
    x_min, y_min = np.nanmin(xy, axis=0)
    x_max, y_max = np.nanmax(xy, axis=0)

    span = max(x_max - x_min, y_max - y_min, 1.0)
    pad = pad_ratio * span

    ax.set_xlim(x_min - pad, x_max + pad)
    ax.set_ylim(y_min - pad, y_max + pad)

def _extent_limits(records, pad_ratio=0.35, min_span=2.0):
    object_xy = [np.asarray(record["xy"], dtype=float) for record in records if len(record["xy"]) > 0]
    if not object_xy:
        return (-0.5 * min_span, 0.5 * min_span, -0.5 * min_span, 0.5 * min_span)

    xy = np.vstack(object_xy)
    x_min, y_min = np.nanmin(xy, axis=0)
    x_max, y_max = np.nanmax(xy, axis=0)

    x_center = 0.5 * (x_min + x_max)
    y_center = 0.5 * (y_min + y_max)

    span = max(x_max - x_min, y_max - y_min, min_span)
    half_span = 0.5 * span * (1.0 + 2.0 * pad_ratio)

    return (
        x_center - half_span,
        x_center + half_span,
        y_center - half_span,
        y_center + half_span,
    )

def _resolve_crange(magnitudes, crange):
    finite = np.asarray(magnitudes, dtype=float)
    finite = finite[np.isfinite(finite)]

    if crange is not None:
        resolved = float(crange)
        return resolved if resolved > 0 else 1.0

    if len(finite) == 0:
        return 1.0

    auto_crange = float(np.nanmax(np.abs(finite)))
    return auto_crange if auto_crange > 0 else 1.0

def _plot_spatial_panel(
    ax,
    records,
    background_record=None,
    title="",
    crange=None,
    zoom=False,
    add_colorbar=True,
    show_legend=True,
    legend_loc="upper right",
    legend_bbox_to_anchor=None,
    colorbar_ax=None,
):
    if background_record is None:
        background_record = next(
            (record for record in records if record["name"] == "GT"),
            records[0],
        )

    cloud, magnitudes = _raw_changemap(background_record)
    resolved_crange = _resolve_crange(magnitudes, crange)

    scatter = ax.scatter(
        cloud[:, 0],
        cloud[:, 1],
        c=magnitudes,
        cmap="seismic_r",
        vmin=-resolved_crange,
        vmax=resolved_crange,
        s=1.0,
        linewidths=0,
        alpha=0.85,
        zorder=1,
    )

    for record in records:
        _draw_object_extent(
            ax,
            record["xy"],
            color=record["color"],
            label=f"{record['name']} object {record['object_id']}",
            linewidth=2.6 if record["name"] == "GT" else 2.2,
            point_size=13 if record["name"] == "GT" else 10,
        )

    if zoom:
        x_min, x_max, y_min, y_max = _extent_limits(records)
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)
    else:
        _scene_limits(ax, cloud)

    ax.set_title(title)
    ax.set_xlabel("X [m]")
    ax.set_ylabel("Y [m]")
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.25)

    if show_legend:
        ax.legend(
            loc=legend_loc,
            bbox_to_anchor=legend_bbox_to_anchor,
            framealpha=0.9,
            borderaxespad=0.2,
        )

    if add_colorbar:
        if colorbar_ax is None:
            cbar = ax.figure.colorbar(
                scatter,
                ax=ax,
                format="%.2f",
                fraction=0.046,
                pad=0.02,
            )
        else:
            cbar = ax.figure.colorbar(
                scatter,
                cax=colorbar_ax,
                format="%.2f",
            )
        cbar.set_label(f"Change magnitude [m], scale +/-{resolved_crange:.2f}")
        cbar.ax._is_object_colorbar = True

    return scatter


In [ ]:
# Temporal interval figure and separate changemap figures.

def _format_timestamp(ts):
    return f"{ts:%Y-%m-%d}"


def _annotate_interval_endpoint(ax, timestamp, y, text, color, ha, x_offset, y_offset):
    ax.annotate(
        text,
        xy=(timestamp, y),
        xytext=(x_offset, y_offset),
        textcoords="offset points",
        ha=ha,
        va="center",
        fontsize=OBJECT_ANNOTATION_SIZE,
        rotation=35,
        color=color,
        clip_on=False,
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.78, pad=0.5),
    )


def plot_single_reference_temporal_intervals(records):
    fig, ax_time = plt.subplots(
        1,
        1,
        figsize=OBJECT_TEMPORAL_FIGSIZE,
        dpi=OBJECT_FIG_DPI,
        constrained_layout=True,
    )

    y_positions = {"GT": 2, "M3C2": 1, "TAM3C2": 0}
    start_offsets = {"GT": 22, "M3C2": 16, "TAM3C2": 10}
    end_offsets = {"GT": -22, "M3C2": -16, "TAM3C2": -10}

    for record in records:
        name = record["name"]
        interval = record["interval"]
        y = y_positions[name]
        color = record["color"]

        start = interval["start_timestamp"]
        end = interval["end_timestamp"]

        ax_time.hlines(
            y,
            start,
            end,
            color=color,
            linewidth=7,
            alpha=0.75,
            label=f"{name} object {record['object_id']}",
        )
        ax_time.scatter([start, end], [y, y], color=color, s=36, zorder=3)

        _annotate_interval_endpoint(
            ax_time,
            start,
            y,
            _format_timestamp(start),
            color=color,
            ha="left",
            x_offset=5,
            y_offset=start_offsets.get(name, 14),
        )
        _annotate_interval_endpoint(
            ax_time,
            end,
            y,
            _format_timestamp(end),
            color=color,
            ha="right",
            x_offset=-5,
            y_offset=end_offsets.get(name, -14),
        )

    if len(records) >= 2:
        overlap_start = max(record["interval"]["start_timestamp"] for record in records)
        overlap_end = min(record["interval"]["end_timestamp"] for record in records)

        if overlap_start <= overlap_end:
            ax_time.axvspan(
                overlap_start,
                overlap_end,
                color="green",
                alpha=0.15,
                label="common overlap",
            )

    ax_time.set_yticks([0, 1, 2])
    ax_time.set_yticklabels(["TAM3C2", "M3C2", "GT"])
    ax_time.set_ylim(-0.65, 2.65)
    ax_time.set_title("Temporal intervals of selected objects")
    ax_time.set_xlabel("Date")
    ax_time.grid(axis="x", alpha=0.25)
    ax_time.margins(x=0.12)

    locator = mdates.AutoDateLocator(minticks=4, maxticks=8)
    ax_time.xaxis.set_major_locator(locator)
    ax_time.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))

    for tick in ax_time.get_xticklabels():
        tick.set_rotation(35)
        tick.set_ha("right")

    ax_time.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.22),
        ncol=min(4, len(records) + 1),
        framealpha=0.9,
    )

    _apply_object_plot_font_sizes(fig)
    plt.show()


def plot_single_reference_changemap(records, crange=None):
    background_record = next(
        (record for record in records if record["name"] == "GT"),
        records[0],
    )

    fig, ax_scene, ax_colorbar = _new_single_changemap_figure()

    _plot_spatial_panel(
        ax_scene,
        records,
        background_record=background_record,
        title="Changemap",
        crange=crange,
        zoom=False,
        add_colorbar=True,
        show_legend=True,
        colorbar_ax=ax_colorbar,
    )
    ax_scene.set_box_aspect(1.0)
    ax_scene.set_anchor("C")

    _apply_object_plot_font_sizes(fig)
    plt.show()


def plot_single_reference_zoom(records, crange=None):
    background_record = next(
        (record for record in records if record["name"] == "GT"),
        records[0],
    )

    fig, ax_zoom, ax_colorbar = _new_single_changemap_figure()

    _plot_spatial_panel(
        ax_zoom,
        records,
        background_record=background_record,
        title="Zoom-in objects",
        crange=crange,
        zoom=True,
        add_colorbar=True,
        show_legend=False,
        colorbar_ax=ax_colorbar,
    )
    ax_zoom.set_box_aspect(1.0)
    ax_zoom.set_anchor("C")

    _apply_object_plot_font_sizes(fig)
    plt.show()


plot_single_reference_temporal_intervals(selected_extent_records)
plot_single_reference_changemap(selected_extent_records, crange=None)
plot_single_reference_zoom(selected_extent_records, crange=None)


In [ ]:
# Raw distance time series of the selected GT/M3C2/TAM3C2 objects.
def _distance_series_for_object_plot(st_analysis, use_smoothed=False):
    if use_smoothed:
        raise ValueError(
            "This evaluation is configured to use raw distances only. "
            "Set use_smoothed_time_series=False."
        )
    return np.asarray(st_analysis.distances, dtype=float), "distances"


def _previous_cell_crange(records, crange=None):
    # Use exactly the same changemap scale logic as the previous cell:
    # background_record = GT if available, otherwise records[0].
    background_record = next(
        (record for record in records if record["name"] == "GT"),
        records[0],
    )
    _, magnitudes = _raw_changemap(background_record)
    return _resolve_crange(magnitudes, crange)


def _shared_time_series_ylim_and_ticks(
    records,
    lower_percentile=1.0,
    upper_percentile=99.0,
    pad_ratio=0.08,
    nbins=5,
):
    all_values = []
    protected_values = [0.0]

    for record in records:
        name = record["name"]
        object_id = record["object_id"]

        st_analysis = analyses[name]
        obj = _analysis_object(name, object_id)
        indices = _object_indices(obj)

        distances, _ = _distance_series_for_object_plot(
            st_analysis,
            use_smoothed=use_smoothed_time_series,
        )

        object_series = np.asarray(distances[indices, :], dtype=float)
        finite = object_series[np.isfinite(object_series)]

        if len(finite):
            all_values.append(finite)

        with np.errstate(all="ignore"):
            mean_series = np.nanmean(object_series, axis=0)
        mean_finite = mean_series[np.isfinite(mean_series)]
        if len(mean_finite):
            protected_values.extend(mean_finite.tolist())

    if not all_values:
        return None, None

    all_values = np.concatenate(all_values)

    y_min = float(np.nanpercentile(all_values, lower_percentile))
    y_max = float(np.nanpercentile(all_values, upper_percentile))

    protected_values = np.asarray(protected_values, dtype=float)
    protected_values = protected_values[np.isfinite(protected_values)]

    if len(protected_values):
        y_min = min(y_min, float(np.nanmin(protected_values)))
        y_max = max(y_max, float(np.nanmax(protected_values)))

    if y_min == y_max:
        pad = max(abs(y_min) * pad_ratio, 0.1)
    else:
        pad = pad_ratio * (y_max - y_min)

    y_min -= pad
    y_max += pad

    locator = MaxNLocator(nbins=nbins)
    ticks = locator.tick_values(y_min, y_max)

    return (float(ticks[0]), float(ticks[-1])), ticks


def _plot_single_method_object_time_series_figure(
    record,
    shared_crange,
    shared_ylim,
    shared_yticks,
):
    name = record["name"]
    object_id = record["object_id"]

    fig, ax_ts, ax_map, ax_colorbar = _new_time_series_changemap_figure()

    st_analysis = analyses[name]
    obj = _analysis_object(name, object_id)
    indices = _object_indices(obj)

    timestamps, _ = _timestamp_info_for(name)
    distances, distance_label = _distance_series_for_object_plot(
        st_analysis,
        use_smoothed=use_smoothed_time_series,
    )

    interval = record["interval"]
    start_epoch = int(interval["start_epoch"])
    end_epoch = int(interval["end_epoch"])

    object_series = distances[indices, :]
    changemap = distances[:, end_epoch] - distances[:, start_epoch]

    cmap = plt.get_cmap("seismic_r").copy()
    norm = plt.Normalize(vmin=-shared_crange, vmax=shared_crange)

    for idx, series in zip(indices, object_series):
        ax_ts.plot(
            timestamps,
            series,
            color=cmap(norm(changemap[idx])),
            alpha=0.22,
            linewidth=0.7,
        )

    with np.errstate(all="ignore"):
        mean_series = np.nanmean(object_series, axis=0)

    ax_ts.plot(
        timestamps,
        mean_series,
        color="black",
        linewidth=2.0,
        label="mean object corepoint time series",
    )
    ax_ts.axvspan(
        interval["start_timestamp"],
        interval["end_timestamp"],
        color="grey",
        alpha=0.22,
        label="4D-OBC timespan",
    )
    ax_ts.axhline(0, color="0.5", linewidth=0.8, linestyle="--")

    if shared_ylim is not None:
        ax_ts.set_ylim(*shared_ylim)
        ax_ts.set_yticks(shared_yticks)

    ax_ts.set_xlabel("Date")
    ax_ts.set_ylabel("Distance [m]")
    ax_ts.set_title(
        f"{name} object {object_id}: "
        f"{len(indices)} corepoints, raw {distance_label}, "
        f"{interval['duration_days']:.1f} days"
    )
    ax_ts.grid(alpha=0.25)
    ax_ts.legend(loc="upper right", framealpha=0.9)

    for tick in ax_ts.get_xticklabels():
        tick.set_rotation(35)
        tick.set_ha("right")
    ax_ts.tick_params(axis="both", which="both", labelbottom=True, labelleft=True)

    _plot_spatial_panel(
        ax_map,
        [record],
        background_record=record,
        title=f"{name} changemap",
        crange=shared_crange,
        zoom=False,
        add_colorbar=True,
        show_legend=True,
        colorbar_ax=ax_colorbar,
    )

    ax_map.set_box_aspect(1.0)
    ax_map.set_anchor("C")
    ax_map.tick_params(axis="both", which="both", labelbottom=True, labelleft=True)

    _apply_object_plot_font_sizes(fig)
    plt.show()


def plot_single_reference_object_time_series(records, crange=None):
    shared_crange = _previous_cell_crange(records, crange=crange)
    shared_ylim, shared_yticks = _shared_time_series_ylim_and_ticks(records)

    print(f"Using previous-cell changemap scale for all three panels: +/-{shared_crange:.3f} m")
    if shared_ylim is not None:
        print(
            f"Using robust shared time-series y-axis: "
            f"{shared_ylim[0]:.3f} to {shared_ylim[1]:.3f} m"
        )

    record_by_name = {record["name"]: record for record in records}

    for name in METHODS:
        record = record_by_name.get(name)
        if record is None:
            fig, ax = plt.subplots(
                1,
                1,
                figsize=OBJECT_TIME_SERIES_MAP_FIGSIZE,
                dpi=OBJECT_FIG_DPI,
                constrained_layout=True,
            )
            ax.text(
                0.5,
                0.5,
                f"{name}: no matched object",
                transform=ax.transAxes,
                ha="center",
                va="center",
                fontsize=OBJECT_FONT_SIZE,
            )
            ax.set_axis_off()
            _apply_object_plot_font_sizes(fig)
            plt.show()
            continue

        _plot_single_method_object_time_series_figure(
            record,
            shared_crange,
            shared_ylim,
            shared_yticks,
        )


plot_single_reference_object_time_series(selected_extent_records, crange=None)


# iou relation table
reference_to_detected_iou.csv: for easily checking, on a case-by-case basis, which detections correspond to each ground truth (GT) object;  
detected_to_reference_iou.csv: for easily checking, on a case-by-case basis, which ground truth (GT) objects each detection overlaps with.


In [ ]:
# Complete detected-reference overlap relationships.
# Run this cell after the new continuous-support evaluation cell.

IOU_EPS = 1e-12

required_columns = {
    "method",
    "detected_object_index",
    "reference_object_index",
    "spatial_iou",
    "temporal_iou",
    "spatiotemporal_iou",
    "shared_corepoint_count",
    "overlapping_duration",
}

missing_columns = required_columns - set(pairwise_metrics_df.columns)
if missing_columns:
    raise ValueError(
        f"pairwise_metrics_df is missing columns: {sorted(missing_columns)}"
    )


# ------------------------------------------------------------------
# 1. All object pairs with non-zero joint spatiotemporal overlap
# ------------------------------------------------------------------

overlap_pairs_df = pairwise_metrics_df.loc[
    np.isfinite(pairwise_metrics_df["spatiotemporal_iou"])
    & (pairwise_metrics_df["spatiotemporal_iou"] > IOU_EPS)
].copy()

# Mark pairs selected by the official one-to-one matching.
official_match_keys = matched_objects_df[
    [
        "method",
        "detected_object_index",
        "reference_object_index",
    ]
].drop_duplicates()

official_match_keys["official_one_to_one_match"] = True

overlap_pairs_df = overlap_pairs_df.merge(
    official_match_keys,
    on=[
        "method",
        "detected_object_index",
        "reference_object_index",
    ],
    how="left",
)

overlap_pairs_df["official_one_to_one_match"] = (
    overlap_pairs_df["official_one_to_one_match"]
    .fillna(False)
    .astype(bool)
)

# Add percentage columns for easier inspection.
for column in [
    "spatial_iou",
    "temporal_iou",
    "spatiotemporal_iou",
]:
    overlap_pairs_df[f"{column}_pct"] = (
        100.0 * overlap_pairs_df[column]
    )

relation_columns = [
    "method",
    "reference_object_index",
    "detected_object_index",
    "spatial_iou",
    "temporal_iou",
    "spatiotemporal_iou",
    "spatial_iou_pct",
    "temporal_iou_pct",
    "spatiotemporal_iou_pct",
    "shared_corepoint_count",
    "overlapping_duration",
    "official_one_to_one_match",
]


# ------------------------------------------------------------------
# 2. Reference -> detected direction
# ------------------------------------------------------------------

reference_to_detected_df = (
    overlap_pairs_df[relation_columns]
    .sort_values(
        [
            "method",
            "reference_object_index",
            "spatiotemporal_iou",
        ],
        ascending=[True, True, False],
    )
    .reset_index(drop=True)
)

print("Reference objects -> detected objects with non-zero ST-IoU")
display(reference_to_detected_df)

# Compact index list for every reference object.
reference_index_list_df = (
    reference_to_detected_df
    .groupby(
        ["method", "reference_object_index"],
        as_index=False,
    )
    .agg(
        detected_object_indices=(
            "detected_object_index",
            lambda values: list(map(int, values)),
        ),
        spatial_ious=(
            "spatial_iou",
            lambda values: list(np.round(values, 4)),
        ),
        temporal_ious=(
            "temporal_iou",
            lambda values: list(np.round(values, 4)),
        ),
        spatiotemporal_ious=(
            "spatiotemporal_iou",
            lambda values: list(np.round(values, 4)),
        ),
    )
)

print("Compact reference-centred index lists")
display(reference_index_list_df)


# ------------------------------------------------------------------
# 3. Detected -> reference direction
# ------------------------------------------------------------------

detected_to_reference_df = (
    overlap_pairs_df[relation_columns]
    .sort_values(
        [
            "method",
            "detected_object_index",
            "spatiotemporal_iou",
        ],
        ascending=[True, True, False],
    )
    .reset_index(drop=True)
)

print("Detected objects -> reference objects with non-zero ST-IoU")
display(detected_to_reference_df)

detected_index_list_df = (
    detected_to_reference_df
    .groupby(
        ["method", "detected_object_index"],
        as_index=False,
    )
    .agg(
        reference_object_indices=(
            "reference_object_index",
            lambda values: list(map(int, values)),
        ),
        spatial_ious=(
            "spatial_iou",
            lambda values: list(np.round(values, 4)),
        ),
        temporal_ious=(
            "temporal_iou",
            lambda values: list(np.round(values, 4)),
        ),
        spatiotemporal_ious=(
            "spatiotemporal_iou",
            lambda values: list(np.round(values, 4)),
        ),
    )
)

print("Compact detected-centred index lists")
display(detected_index_list_df)


# ------------------------------------------------------------------
# 4. Detected-object indices with no joint overlap with any reference
# ------------------------------------------------------------------

detected_without_reference_overlap = {}

for method in ["M3C2", "TAM3C2"]:
    detected_with_overlap = set(
        overlap_pairs_df.loc[
            overlap_pairs_df["method"] == method,
            "detected_object_index",
        ].astype(int)
    )

    zero_st_iou_indices = sorted(
        set(range(len(analyses[method].objects)))
        - detected_with_overlap
    )

    detected_without_reference_overlap[method] = (
        zero_st_iou_indices
    )

    print(
        f"{method} detected-object indices with zero ST-IoU "
        "against every reference object:"
    )
    print(zero_st_iou_indices)
    print(f"Length: {len(zero_st_iou_indices)}")
    print()


# ------------------------------------------------------------------
# 5. Save the diagnostic tables
# ------------------------------------------------------------------

diagnostic_output_dir = os.path.join(
    archive_folder,
    "object_overlap_diagnostics",
)
os.makedirs(diagnostic_output_dir, exist_ok=True)

overlap_pairs_df.to_csv(
    os.path.join(
        diagnostic_output_dir,
        "all_nonzero_overlap_pairs.csv",
    ),
    index=False,
)

reference_to_detected_df.to_csv(
    os.path.join(
        diagnostic_output_dir,
        "reference_to_detected_iou.csv",
    ),
    index=False,
)

reference_index_list_df.to_csv(
    os.path.join(
        diagnostic_output_dir,
        "reference_to_detected_index_lists.csv",
    ),
    index=False,
)

detected_to_reference_df.to_csv(
    os.path.join(
        diagnostic_output_dir,
        "detected_to_reference_iou.csv",
    ),
    index=False,
)

detected_index_list_df.to_csv(
    os.path.join(
        diagnostic_output_dir,
        "detected_to_reference_index_lists.csv",
    ),
    index=False,
)

print(f"Saved overlap diagnostics to: {diagnostic_output_dir}")

# figures
- GT-centred paired plot (one-to-one)
- Detected-object classification

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle


METHODS = ["M3C2", "TAM3C2"]
METHOD_COLORS = {
    "M3C2": "#4C78A8",
    "TAM3C2": "#F58518",
}

IOU_EPS = 1e-12

# Used only for diagnosing fragmentation and merging.
# Very small overlaps below this value are ignored when identifying topology.
DIAGNOSTIC_IOU_THRESHOLD = 0.05


required_pairwise_columns = {
    "method",
    "detected_object_index",
    "reference_object_index",
    "spatiotemporal_iou",
}

missing_columns = (
    required_pairwise_columns
    - set(pairwise_metrics_df.columns)
)

if missing_columns:
    raise ValueError(
        "pairwise_metrics_df is missing columns: "
        f"{sorted(missing_columns)}"
    )


pairwise_plot_df = pairwise_metrics_df.copy()

pairwise_plot_df["detected_object_index"] = (
    pairwise_plot_df["detected_object_index"].astype(int)
)

pairwise_plot_df["reference_object_index"] = (
    pairwise_plot_df["reference_object_index"].astype(int)
)

pairwise_plot_df["spatiotemporal_iou"] = (
    pd.to_numeric(
        pairwise_plot_df["spatiotemporal_iou"],
        errors="coerce",
    )
    .fillna(0.0)
    .clip(0.0, 1.0)
)


n_reference_objects = len(analyses["GT"].objects)
reference_indices = np.arange(n_reference_objects)


# matched_objects_df contains only successful official one-to-one matches
# that already satisfy match_iou_threshold.
if (
    isinstance(matched_objects_df, pd.DataFrame)
    and len(matched_objects_df) > 0
):
    official_matches_df = matched_objects_df[
        [
            "method",
            "detected_object_index",
            "reference_object_index",
            "spatiotemporal_iou",
        ]
    ].copy()

    official_matches_df["detected_object_index"] = (
        official_matches_df["detected_object_index"].astype(int)
    )

    official_matches_df["reference_object_index"] = (
        official_matches_df["reference_object_index"].astype(int)
    )

else:
    official_matches_df = pd.DataFrame(
        columns=[
            "method",
            "detected_object_index",
            "reference_object_index",
            "spatiotemporal_iou",
        ]
    )


print(
    f"Reference objects: {n_reference_objects}"
)

for method in METHODS:
    print(
        f"{method} detected objects: "
        f"{len(analyses[method].objects)}"
    )

In [ ]:
# Recommended for the main manuscript figure:
#ORDER_MODE = "reference_index"
ORDER_MODE = "tam3c2_descending"

In [ ]:
# ================================================================
# Plot configuration
# ================================================================

if ORDER_MODE == "reference_index":
    ordered_reference_indices = (
        reference_indices.copy()
    )

    x_axis_label = (
        "Reference-object index"
    )

    plot_title = (
        "GT-centred one-to-one assignment comparison"
    )

elif ORDER_MODE == "tam3c2_descending":
    tam3c2_sort_values = np.nan_to_num(
        tam3c2_iou,
        nan=-np.inf,
    )

    ordered_reference_indices = np.argsort(
        -tam3c2_sort_values
    )

    x_axis_label = (
        "Reference objects ordered by "
        "TAM3C2 assigned ST-IoU"
    )

    plot_title = (
        "GT-centred one-to-one assignment comparison\n"
        "Reference objects ordered by TAM3C2 performance"
    )

else:
    raise ValueError(
        "ORDER_MODE must be either "
        "'reference_index' or "
        "'tam3c2_descending'."
    )


# Plot positions are ranks/positions on the figure.
plot_positions = np.arange(
    n_reference_objects
)

ordered_m3c2_iou = m3c2_iou[
    ordered_reference_indices
]

ordered_tam3c2_iou = tam3c2_iou[
    ordered_reference_indices
]


# ================================================================
# Plot
# ================================================================

fig_width = max(
    11,
    0.42 * n_reference_objects,
)

fig, ax = plt.subplots(
    figsize=(fig_width, 6.5),
    dpi=130,
    constrained_layout=True,
)


# ------------------------------------------------------------
# Vertical paired connectors
# ------------------------------------------------------------

for position, m3c2_value, tam3c2_value in zip(
    plot_positions,
    ordered_m3c2_iou,
    ordered_tam3c2_iou,
):
    if (
        np.isfinite(m3c2_value)
        and np.isfinite(tam3c2_value)
    ):
        ax.vlines(
            x=position,
            ymin=min(
                m3c2_value,
                tam3c2_value,
            ),
            ymax=max(
                m3c2_value,
                tam3c2_value,
            ),
            color="0.68",
            alpha=0.75,
            linewidth=1.4,
            zorder=1,
        )


# ------------------------------------------------------------
# M3C2: all assigned objects are solid blue circles
# ------------------------------------------------------------

m3c2_finite = np.isfinite(
    ordered_m3c2_iou
)

ax.scatter(
    plot_positions[m3c2_finite],
    ordered_m3c2_iou[m3c2_finite],
    s=68,
    marker="o",
    color=METHOD_COLORS["M3C2"],
    edgecolor="white",
    linewidth=0.7,
    label="M3C2 assigned object",
    zorder=3,
)


# ------------------------------------------------------------
# TAM3C2: all assigned objects are solid orange diamonds
# ------------------------------------------------------------

tam3c2_finite = np.isfinite(
    ordered_tam3c2_iou
)

ax.scatter(
    plot_positions[tam3c2_finite],
    ordered_tam3c2_iou[tam3c2_finite],
    s=53,
    marker="D",
    color=METHOD_COLORS["TAM3C2"],
    edgecolor="white",
    linewidth=0.7,
    label="TAM3C2 assigned object",
    zorder=4,
)


# ------------------------------------------------------------
# References without assignment, if any
# ------------------------------------------------------------

m3c2_unassigned = ~m3c2_finite

if np.any(m3c2_unassigned):
    ax.scatter(
        plot_positions[m3c2_unassigned],
        np.zeros(
            m3c2_unassigned.sum()
        ),
        s=55,
        marker="x",
        color=METHOD_COLORS["M3C2"],
        linewidth=1.5,
        label="M3C2 unassigned reference",
        zorder=4,
    )


tam3c2_unassigned = ~tam3c2_finite

if np.any(tam3c2_unassigned):
    ax.scatter(
        plot_positions[tam3c2_unassigned],
        np.zeros(
            tam3c2_unassigned.sum()
        ),
        s=55,
        marker="x",
        color=METHOD_COLORS["TAM3C2"],
        linewidth=1.5,
        label="TAM3C2 unassigned reference",
        zorder=4,
    )


# ------------------------------------------------------------
# Match threshold
# ------------------------------------------------------------

ax.axhline(
    MATCH_THRESHOLD,
    color="black",
    linestyle="--",
    linewidth=1.1,
    label=(
        f"Match threshold = "
        f"{MATCH_THRESHOLD:.2f}"
    ),
)


ax.set_xlabel(
    x_axis_label
)

ax.set_ylabel(
    "Assigned spatiotemporal IoU"
)

ax.set_title(
    plot_title
)


# Tick positions are plot positions.
# Tick labels retain the real reference-object indices.
ax.set_xticks(
    plot_positions
)

ax.set_xticklabels(
    ordered_reference_indices
)

if n_reference_objects > 25:
    plt.setp(
        ax.get_xticklabels(),
        rotation=45,
        ha="right",
    )


ax.set_ylim(
    -0.03,
    1.03,
)

ax.grid(
    axis="y",
    alpha=0.25,
)

ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.14),
    ncol=2,
    frameon=False,
)

plt.show()

In [ ]:
# ================================================================
# Print reference-object counts and percentages
# ================================================================

def print_reference_result(
    label,
    object_indices,
    total_reference_objects,
):
    object_indices = list(
        map(int, object_indices)
    )

    count = len(object_indices)

    percentage = (
        100.0
        * count
        / total_reference_objects
        if total_reference_objects > 0
        else np.nan
    )

    print(
        f"{label}: "
        f"{count}/{total_reference_objects} "
        f"({percentage:.1f}%)"
    )

    print(object_indices)
    print()


print(
    "GT-centred one-to-one comparison"
)

print("-" * 65)


print_reference_result(
    "Only TAM3C2 successful",
    only_tam3c2_success,
    n_reference_objects,
)

print_reference_result(
    "Only M3C2 successful",
    only_m3c2_success,
    n_reference_objects,
)

print_reference_result(
    "Both methods successful",
    both_successful,
    n_reference_objects,
)

print_reference_result(
    "Neither method successful",
    neither_successful,
    n_reference_objects,
)

print_reference_result(
    "TAM3C2 has higher assigned ST-IoU",
    tam3c2_higher_assigned_iou,
    n_reference_objects,
)

print_reference_result(
    "M3C2 has higher assigned ST-IoU",
    m3c2_higher_assigned_iou,
    n_reference_objects,
)

In [ ]:
comparison_summary_rows = []


summary_groups = {
    "Only TAM3C2 successful": (
        only_tam3c2_success
    ),
    "Only M3C2 successful": (
        only_m3c2_success
    ),
    "Both methods successful": (
        both_successful
    ),
    "Neither method successful": (
        neither_successful
    ),
    "TAM3C2 has higher assigned ST-IoU": (
        tam3c2_higher_assigned_iou
    ),
    "M3C2 has higher assigned ST-IoU": (
        m3c2_higher_assigned_iou
    ),
}


for result_type, object_indices in (
    summary_groups.items()
):
    count = len(object_indices)

    percentage = (
        100.0
        * count
        / n_reference_objects
        if n_reference_objects > 0
        else np.nan
    )

    comparison_summary_rows.append(
        {
            "result_type": result_type,
            "count": count,
            "total_reference_objects": (
                n_reference_objects
            ),
            "percentage_of_references": (
                percentage
            ),
            "reference_object_indices": (
                list(
                    map(
                        int,
                        object_indices,
                    )
                )
            ),
        }
    )


gt_one_to_one_summary_df = (
    pd.DataFrame(
        comparison_summary_rows
    )
)


display(
    gt_one_to_one_summary_df.style.format(
        {
            "percentage_of_references": (
                "{:.1f}%"
            )
        }
    )
)

In [ ]:
# ================================================================
# Configuration
# ================================================================

METHODS = ["M3C2", "TAM3C2"]

IOU_EPS = 1e-12

# Same threshold as the formal object matching.
MATCH_THRESHOLD = match_iou_threshold

# Any positive joint ST overlap is treated as a topology edge.
# If tiny numerical overlaps create too many false topology links,
# test 0.01 or 0.05 instead.
TOPOLOGY_IOU_THRESHOLD = IOU_EPS


CATEGORY_ORDER = [
    "Clean one-to-one, IoU >= 0.8",
    "Clean one-to-one, IoU < 0.8",
    "Fragmentation only",
    "Merging only",
    "Fragmentation + merging",
    "Zero joint ST overlap",
]

CATEGORY_COLORS = {
    "Clean one-to-one, IoU >= 0.8": "#54A24B",
    "Clean one-to-one, IoU < 0.8": "#9EC5AB",
    "Fragmentation only": "#B279A2",
    "Merging only": "#FF9DA6",
    "Fragmentation + merging": "#ECA82C",
    "Zero joint ST overlap": "#E45756",
}


required_columns = {
    "method",
    "detected_object_index",
    "reference_object_index",
    "spatial_iou",
    "temporal_iou",
    "spatiotemporal_iou",
}

missing_columns = (
    required_columns - set(pairwise_metrics_df.columns)
)

if missing_columns:
    raise ValueError(
        "pairwise_metrics_df is missing columns: "
        f"{sorted(missing_columns)}"
    )


pairwise_topology_df = pairwise_metrics_df.copy()

pairwise_topology_df["detected_object_index"] = (
    pairwise_topology_df[
        "detected_object_index"
    ].astype(int)
)

pairwise_topology_df["reference_object_index"] = (
    pairwise_topology_df[
        "reference_object_index"
    ].astype(int)
)

for column in [
    "spatial_iou",
    "temporal_iou",
    "spatiotemporal_iou",
]:
    pairwise_topology_df[column] = (
        pd.to_numeric(
            pairwise_topology_df[column],
            errors="coerce",
        )
        .fillna(0.0)
        .clip(0.0, 1.0)
    )


# ================================================================
# Helper functions
# ================================================================

def _component_pattern(
    spatial_iou,
    temporal_iou,
    threshold=MATCH_THRESHOLD,
):
    spatial_high = spatial_iou >= threshold
    temporal_high = temporal_iou >= threshold

    if spatial_high and temporal_high:
        return "Spatial high; temporal high"

    if spatial_high and not temporal_high:
        return "Spatial high; temporal low"

    if not spatial_high and temporal_high:
        return "Spatial low; temporal high"

    return "Spatial low; temporal low"


def _limiting_dimension(
    spatial_iou,
    temporal_iou,
    tolerance=1e-12,
):
    difference = spatial_iou - temporal_iou

    if difference > tolerance:
        return "Temporal overlap lower"

    if difference < -tolerance:
        return "Spatial overlap lower"

    return "Spatial and temporal equal"


def _positive_iou_list(
    rows,
    value_column,
):
    positive_rows = rows.loc[
        rows[value_column] > IOU_EPS,
        [
            "reference_object_index",
            value_column,
        ],
    ].sort_values(
        value_column,
        ascending=False,
    )

    return [
        (
            int(row.reference_object_index),
            round(float(getattr(row, value_column)), 4),
        )
        for row in positive_rows.itertuples(index=False)
    ]


# ================================================================
# Build topology and mutually exclusive object categories
# ================================================================

classification_rows = []
fragment_pair_rows = []
merged_pair_rows = []
zero_st_rows = []


for method in METHODS:
    method_pairs = pairwise_topology_df.loc[
        pairwise_topology_df["method"] == method
    ].copy()

    n_detected = len(analyses[method].objects)

    topology_edges = method_pairs.loc[
        method_pairs["spatiotemporal_iou"]
        > TOPOLOGY_IOU_THRESHOLD
    ].copy()


    # ------------------------------------------------------------
    # Topology degrees
    # ------------------------------------------------------------

    references_by_detection = (
        topology_edges
        .groupby("detected_object_index")[
            "reference_object_index"
        ]
        .agg(
            lambda values: set(map(int, values))
        )
        .to_dict()
    )

    detections_by_reference = (
        topology_edges
        .groupby("reference_object_index")[
            "detected_object_index"
        ]
        .agg(
            lambda values: set(map(int, values))
        )
        .to_dict()
    )


    # ------------------------------------------------------------
    # Store pair-level fragment and merged diagnostics
    # ------------------------------------------------------------

    for pair in topology_edges.itertuples(index=False):
        detected_idx = int(
            pair.detected_object_index
        )
        reference_idx = int(
            pair.reference_object_index
        )

        spatial_iou = float(pair.spatial_iou)
        temporal_iou = float(pair.temporal_iou)
        st_iou = float(pair.spatiotemporal_iou)

        detected_degree = len(
            references_by_detection.get(
                detected_idx,
                set(),
            )
        )

        reference_degree = len(
            detections_by_reference.get(
                reference_idx,
                set(),
            )
        )

        is_fragment_relation = (
            reference_degree > 1
        )

        is_merged_relation = (
            detected_degree > 1
        )

        common_row = {
            "method": method,
            "detected_object_index": detected_idx,
            "reference_object_index": reference_idx,
            "spatial_iou": spatial_iou,
            "temporal_iou": temporal_iou,
            "spatiotemporal_iou": st_iou,
            "n_references_for_detection": (
                detected_degree
            ),
            "n_detections_for_reference": (
                reference_degree
            ),
            "component_pattern": (
                _component_pattern(
                    spatial_iou,
                    temporal_iou,
                )
            ),
            "limiting_dimension": (
                _limiting_dimension(
                    spatial_iou,
                    temporal_iou,
                )
            ),
        }

        if is_fragment_relation:
            fragment_pair_rows.append(
                common_row.copy()
            )

        if is_merged_relation:
            merged_pair_rows.append(
                common_row.copy()
            )


    # ------------------------------------------------------------
    # Classify every detected object exactly once
    # ------------------------------------------------------------

    for detected_idx in range(n_detected):
        detected_pairs = method_pairs.loc[
            method_pairs["detected_object_index"]
            == detected_idx
        ]

        connected_references = (
            references_by_detection.get(
                detected_idx,
                set(),
            )
        )

        detected_degree = len(
            connected_references
        )


        # Find the best reference for reporting.
        if len(detected_pairs) > 0:
            best_pair_idx = (
                detected_pairs[
                    "spatiotemporal_iou"
                ].idxmax()
            )
            best_pair = detected_pairs.loc[
                best_pair_idx
            ]

            best_reference_idx = int(
                best_pair[
                    "reference_object_index"
                ]
            )

            best_spatial_iou = float(
                best_pair["spatial_iou"]
            )

            best_temporal_iou = float(
                best_pair["temporal_iou"]
            )

            best_st_iou = float(
                best_pair[
                    "spatiotemporal_iou"
                ]
            )

        else:
            best_reference_idx = np.nan
            best_spatial_iou = 0.0
            best_temporal_iou = 0.0
            best_st_iou = 0.0


        # --------------------------------------------------------
        # Zero joint ST overlap
        # --------------------------------------------------------
# --------------------------------------------------------
# Zero joint ST overlap
#
# Each detected object is assigned to exactly one of:
# 1. Spatial overlap only
# 2. Temporal overlap only
# 3. No spatial or temporal overlap
#
# If spatial and temporal partial overlaps exist against
# different references, use the stronger partial overlap
# as the object's primary diagnostic category.
# --------------------------------------------------------

        if detected_degree == 0:
            category = "Zero joint ST overlap"

            if len(detected_pairs) > 0:
                max_spatial_iou = float(
                    detected_pairs["spatial_iou"].max()
                )

                max_temporal_iou = float(
                    detected_pairs["temporal_iou"].max()
                )

            else:
                max_spatial_iou = 0.0
                max_temporal_iou = 0.0


            # ----------------------------------------------------
            # Case 1: neither spatial nor temporal overlap
            # ----------------------------------------------------

            if (
                max_spatial_iou <= IOU_EPS
                and max_temporal_iou <= IOU_EPS
            ):
                zero_reason = (
                    "No spatial or temporal overlap"
                )

                dominant_reference_idx = np.nan
                diagnostic_spatial_iou = 0.0
                diagnostic_temporal_iou = 0.0
                diagnostic_overlap_value = 0.0


            # ----------------------------------------------------
            # Case 2: spatial overlap is the dominant partial match
            # ----------------------------------------------------

            elif (
                max_spatial_iou
                >= max_temporal_iou
            ):
                zero_reason = "Spatial overlap only"

                best_spatial_row_idx = (
                    detected_pairs[
                        "spatial_iou"
                    ].idxmax()
                )

                best_spatial_row = (
                    detected_pairs.loc[
                        best_spatial_row_idx
                    ]
                )

                dominant_reference_idx = int(
                    best_spatial_row[
                        "reference_object_index"
                    ]
                )

                diagnostic_spatial_iou = float(
                    best_spatial_row[
                        "spatial_iou"
                    ]
                )

                diagnostic_temporal_iou = float(
                    best_spatial_row[
                        "temporal_iou"
                    ]
                )

                diagnostic_overlap_value = (
                    diagnostic_spatial_iou
                )


            # ----------------------------------------------------
            # Case 3: temporal overlap is the dominant partial match
            # ----------------------------------------------------

            else:
                zero_reason = "Temporal overlap only"

                best_temporal_row_idx = (
                    detected_pairs[
                        "temporal_iou"
                    ].idxmax()
                )

                best_temporal_row = (
                    detected_pairs.loc[
                        best_temporal_row_idx
                    ]
                )

                dominant_reference_idx = int(
                    best_temporal_row[
                        "reference_object_index"
                    ]
                )

                diagnostic_spatial_iou = float(
                    best_temporal_row[
                        "spatial_iou"
                    ]
                )

                diagnostic_temporal_iou = float(
                    best_temporal_row[
                        "temporal_iou"
                    ]
                )

                diagnostic_overlap_value = (
                    diagnostic_temporal_iou
                )


            zero_st_rows.append(
                {
                    "method": method,
                    "detected_object_index": (
                        detected_idx
                    ),
                    "zero_st_reason": (
                        zero_reason
                    ),
                    "dominant_reference_object_index": (
                        dominant_reference_idx
                    ),
                    "spatial_iou": (
                        diagnostic_spatial_iou
                    ),
                    "temporal_iou": (
                        diagnostic_temporal_iou
                    ),
                    "diagnostic_overlap_value": (
                        diagnostic_overlap_value
                    ),
                    "maximum_spatial_iou_against_all_references": (
                        max_spatial_iou
                    ),
                    "maximum_temporal_iou_against_all_references": (
                        max_temporal_iou
                    ),
                }
            )

            is_fragment_candidate = False
            is_merged_candidate = False

        # --------------------------------------------------------
        # Positive joint ST overlap
        # --------------------------------------------------------

        else:
            connected_reference_degrees = [
                len(
                    detections_by_reference.get(
                        reference_idx,
                        set(),
                    )
                )
                for reference_idx
                in connected_references
            ]

            is_fragment_candidate = any(
                degree > 1
                for degree
                in connected_reference_degrees
            )

            is_merged_candidate = (
                detected_degree > 1
            )


            # Clean topology:
            # detection degree = 1
            # corresponding reference degree = 1
            is_clean_one_to_one = (
                detected_degree == 1
                and not is_fragment_candidate
            )


            if is_clean_one_to_one:
                if best_st_iou >= MATCH_THRESHOLD:
                    category = (
                        "Clean one-to-one, "
                        "IoU >= 0.8"
                    )
                else:
                    category = (
                        "Clean one-to-one, "
                        "IoU < 0.8"
                    )

            elif (
                is_fragment_candidate
                and is_merged_candidate
            ):
                category = (
                    "Fragmentation + merging"
                )

            elif is_fragment_candidate:
                category = (
                    "Fragmentation only"
                )

            elif is_merged_candidate:
                category = "Merging only"

            else:
                raise AssertionError(
                    "Positive-overlap object could "
                    "not be classified."
                )


        classification_rows.append(
            {
                "method": method,
                "detected_object_index": (
                    detected_idx
                ),
                "primary_category": category,
                "best_reference_object_index": (
                    best_reference_idx
                ),
                "best_spatial_iou": (
                    best_spatial_iou
                ),
                "best_temporal_iou": (
                    best_temporal_iou
                ),
                "best_spatiotemporal_iou": (
                    best_st_iou
                ),
                "n_overlapping_references": (
                    detected_degree
                ),
                "is_fragment_candidate": (
                    is_fragment_candidate
                ),
                "is_merged_candidate": (
                    is_merged_candidate
                ),
            }
        )


# ================================================================
# Classification table and count verification
# ================================================================

detected_topology_classification_df = (
    pd.DataFrame(classification_rows)
)


classification_counts_df = (
    detected_topology_classification_df
    .groupby(
        [
            "method",
            "primary_category",
        ]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        index=METHODS,
        columns=CATEGORY_ORDER,
        fill_value=0,
    )
)


classification_counts_df[
    "Classified total"
] = classification_counts_df.sum(axis=1)

classification_counts_df[
    "Detected-object total"
] = [
    len(analyses[method].objects)
    for method in METHODS
]


for method in METHODS:
    classified_total = int(
        classification_counts_df.loc[
            method,
            "Classified total",
        ]
    )

    detected_total = int(
        classification_counts_df.loc[
            method,
            "Detected-object total",
        ]
    )

    if classified_total != detected_total:
        raise AssertionError(
            f"{method}: category counts sum to "
            f"{classified_total}, but the method "
            f"detected {detected_total} objects."
        )


print(
    "Category counts; the classified total must "
    "equal the detected-object total"
)

display(classification_counts_df)


# ================================================================
# Stacked bar
# ================================================================

fig, ax = plt.subplots(
    figsize=(10, 6.5),
    dpi=130,
    constrained_layout=True,
)

bottom = np.zeros(
    len(METHODS),
    dtype=float,
)


for category in CATEGORY_ORDER:
    values = (
        classification_counts_df[
            category
        ].to_numpy(dtype=float)
    )

    bars = ax.bar(
        METHODS,
        values,
        bottom=bottom,
        label=category,
        color=CATEGORY_COLORS[category],
        edgecolor="white",
        linewidth=0.8,
    )

    for bar, value, lower in zip(
        bars,
        values,
        bottom,
    ):
        if value > 0:
            ax.text(
                bar.get_x()
                + bar.get_width() / 2,
                lower + value / 2,
                str(int(value)),
                ha="center",
                va="center",
                fontsize=9,
            )

    bottom += values


# Total-object labels above bars.
for method_idx, method in enumerate(METHODS):
    total = len(
        analyses[method].objects
    )

    ax.text(
        method_idx,
        total + max(1.0, 0.015 * total),
        f"Total = {total}",
        ha="center",
        va="bottom",
        fontweight="bold",
    )


ax.set_ylabel(
    "Number of detected objects"
)

ax.set_title(
    "Detected-object overlap topology"
)

ax.set_ylim(
    0,
    max(bottom) * 1.10
)

ax.grid(
    axis="y",
    alpha=0.22,
)

ax.set_axisbelow(True)

ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.12),
    ncol=2,
    frameon=False,
)

plt.show()


# Index lists for each mutually exclusive category.
category_index_lists_df = (
    detected_topology_classification_df
    .groupby(
        [
            "method",
            "primary_category",
        ]
    )["detected_object_index"]
    .agg(
        lambda values: sorted(
            map(int, values)
        )
    )
    .reset_index(
        name="detected_object_indices"
    )
)

category_index_lists_df["count"] = (
    category_index_lists_df[
        "detected_object_indices"
    ].apply(len)
)

display(category_index_lists_df)


# ================================================================
# Fragment diagnostic tables
# ================================================================

fragment_columns = [
    "method",
    "detected_object_index",
    "reference_object_index",
    "spatial_iou",
    "temporal_iou",
    "spatiotemporal_iou",
    "n_references_for_detection",
    "n_detections_for_reference",
    "component_pattern",
    "limiting_dimension",
]

fragment_overlap_detail_df = pd.DataFrame(
    fragment_pair_rows,
    columns=fragment_columns,
).sort_values(
    [
        "method",
        "reference_object_index",
        "spatiotemporal_iou",
    ],
    ascending=[True, True, False],
).reset_index(drop=True)


print(
    "Fragmentation relations: pair-level "
    "spatial and temporal overlap"
)

display(fragment_overlap_detail_df)


if len(fragment_overlap_detail_df) > 0:
    fragment_overlap_summary_df = (
        fragment_overlap_detail_df
        .groupby("method")
        .agg(
            n_fragment_detections=(
                "detected_object_index",
                "nunique",
            ),
            n_fragmented_references=(
                "reference_object_index",
                "nunique",
            ),
            n_fragment_relations=(
                "spatiotemporal_iou",
                "size",
            ),
            mean_spatial_iou=(
                "spatial_iou",
                "mean",
            ),
            median_spatial_iou=(
                "spatial_iou",
                "median",
            ),
            mean_temporal_iou=(
                "temporal_iou",
                "mean",
            ),
            median_temporal_iou=(
                "temporal_iou",
                "median",
            ),
            mean_spatiotemporal_iou=(
                "spatiotemporal_iou",
                "mean",
            ),
        )
        .reindex(METHODS)
        .fillna(0)
    )

    fragment_pattern_counts_df = (
        fragment_overlap_detail_df
        .groupby(
            [
                "method",
                "component_pattern",
            ]
        )
        .size()
        .unstack(fill_value=0)
        .reindex(index=METHODS, fill_value=0)
    )

else:
    fragment_overlap_summary_df = (
        pd.DataFrame(index=METHODS)
    )
    fragment_pattern_counts_df = (
        pd.DataFrame(index=METHODS)
    )


print(
    "Fragmentation summary"
)

display(fragment_overlap_summary_df)

print(
    "Fragmentation: spatial/temporal "
    "high-low pattern counts"
)

display(fragment_pattern_counts_df)


# ================================================================
# Merged-object diagnostic tables
# ================================================================

merged_overlap_detail_df = pd.DataFrame(
    merged_pair_rows,
    columns=fragment_columns,
).sort_values(
    [
        "method",
        "detected_object_index",
        "spatiotemporal_iou",
    ],
    ascending=[True, True, False],
).reset_index(drop=True)


print(
    "Merging relations: pair-level "
    "spatial and temporal overlap"
)

display(merged_overlap_detail_df)


if len(merged_overlap_detail_df) > 0:
    merged_overlap_summary_df = (
        merged_overlap_detail_df
        .groupby("method")
        .agg(
            n_merged_detections=(
                "detected_object_index",
                "nunique",
            ),
            n_affected_references=(
                "reference_object_index",
                "nunique",
            ),
            n_merged_relations=(
                "spatiotemporal_iou",
                "size",
            ),
            mean_spatial_iou=(
                "spatial_iou",
                "mean",
            ),
            median_spatial_iou=(
                "spatial_iou",
                "median",
            ),
            mean_temporal_iou=(
                "temporal_iou",
                "mean",
            ),
            median_temporal_iou=(
                "temporal_iou",
                "median",
            ),
            mean_spatiotemporal_iou=(
                "spatiotemporal_iou",
                "mean",
            ),
        )
        .reindex(METHODS)
        .fillna(0)
    )

    merged_pattern_counts_df = (
        merged_overlap_detail_df
        .groupby(
            [
                "method",
                "component_pattern",
            ]
        )
        .size()
        .unstack(fill_value=0)
        .reindex(index=METHODS, fill_value=0)
    )

else:
    merged_overlap_summary_df = (
        pd.DataFrame(index=METHODS)
    )
    merged_pattern_counts_df = (
        pd.DataFrame(index=METHODS)
    )


print(
    "Merged-object summary"
)

display(merged_overlap_summary_df)

print(
    "Merging: spatial/temporal "
    "high-low pattern counts"
)

display(merged_pattern_counts_df)

# ================================================================
# Zero joint ST-overlap diagnostic tables
#
# Three mutually exclusive reasons:
# 1. Spatial overlap only
# 2. Temporal overlap only
# 3. No spatial or temporal overlap
# ================================================================

ZERO_REASON_ORDER = [
    "Spatial overlap only",
    "Temporal overlap only",
    "No spatial or temporal overlap",
]


zero_st_overlap_detail_df = (
    pd.DataFrame(zero_st_rows)
    .sort_values(
        [
            "method",
            "zero_st_reason",
            "diagnostic_overlap_value",
        ],
        ascending=[True, True, False],
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Summary counts
# ------------------------------------------------------------

zero_st_overlap_counts_df = (
    zero_st_overlap_detail_df
    .groupby(
        [
            "method",
            "zero_st_reason",
        ]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        index=METHODS,
        columns=ZERO_REASON_ORDER,
        fill_value=0,
    )
)


zero_st_overlap_counts_df[
    "Zero-ST total"
] = zero_st_overlap_counts_df.sum(
    axis=1
)


# Verify against the main object classification.
expected_zero_counts = (
    detected_topology_classification_df.loc[
        detected_topology_classification_df[
            "primary_category"
        ] == "Zero joint ST overlap"
    ]
    .groupby("method")
    .size()
    .reindex(
        METHODS,
        fill_value=0,
    )
)


for method in METHODS:
    diagnosed_total = int(
        zero_st_overlap_counts_df.loc[
            method,
            "Zero-ST total",
        ]
    )

    expected_total = int(
        expected_zero_counts.loc[method]
    )

    if diagnosed_total != expected_total:
        raise AssertionError(
            f"{method}: the three zero-ST categories "
            f"contain {diagnosed_total} objects, "
            f"but {expected_total} zero-ST objects "
            "were expected."
        )


print(
    "Zero joint ST-overlap object counts\n"
    "The three categories are mutually exclusive."
)

display(zero_st_overlap_counts_df)


# ------------------------------------------------------------
# Object-index and IoU lists
# ------------------------------------------------------------

zero_st_overlap_lists_df = (
    zero_st_overlap_detail_df
    .groupby(
        [
            "method",
            "zero_st_reason",
        ]
    )
    .agg(
        detected_object_indices=(
            "detected_object_index",
            lambda values: list(
                map(int, values)
            ),
        ),
        dominant_reference_indices=(
            "dominant_reference_object_index",
            lambda values: [
                (
                    int(value)
                    if np.isfinite(value)
                    else None
                )
                for value in values
            ],
        ),
        spatial_iou_values=(
            "spatial_iou",
            lambda values: list(
                np.round(values, 4)
            ),
        ),
        temporal_iou_values=(
            "temporal_iou",
            lambda values: list(
                np.round(values, 4)
            ),
        ),
    )
    .reset_index()
)


zero_st_overlap_lists_df["count"] = (
    zero_st_overlap_lists_df[
        "detected_object_indices"
    ].apply(len)
)


print(
    "Zero-ST object indices and partial-overlap IoUs"
)

display(zero_st_overlap_lists_df)


# ------------------------------------------------------------
# Detailed object-level table
# ------------------------------------------------------------

print(
    "Detailed zero-ST object diagnostics"
)

display(
    zero_st_overlap_detail_df[
        [
            "method",
            "detected_object_index",
            "zero_st_reason",
            "dominant_reference_object_index",
            "spatial_iou",
            "temporal_iou",
            "diagnostic_overlap_value",
        ]
    ]
)